In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 1 — ENVIRONMENT SETUP & RAW DATA INGESTION
# ------------------------------------------------------------------------------
# Purpose : Mount Google Drive, discover all 28 quarterly FAERS ASCII files
#           (7 file types x 4 quarters), load them in their original raw text
#           form with no manual pre-cleaning, and concatenate each file type
#           into a single full-year dataframe.
# Output  : 7 raw, uncleaned, full-year dataframes (one per FAERS file type),
#           cached to disk as pickles so later steps don't need to re-parse
#           28 text files every time the Colab runtime restarts.
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import glob
import pickle
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")  # FAERS files throw benign dtype-inference warnings; suppressed after review

Mounted at /content/drive


In [ ]:
# EDIT THIS: point to the folder in your Drive that contains the FAERS files
# (searches subfolders too, so it doesn't matter if each quarter is in its own
# sub-directory, e.g. .../FAERS2025/ASCII_2025Q1/, .../ASCII_2025Q2/, etc.)
BASE_DIR = "/content/drive/MyDrive/FAERS 2025"

# Where cleaned/cached intermediate outputs get written, kept separate from
# raw source files so raw data is never accidentally overwritten.
CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# The 7 official FAERS file types (Section C of ASC_NTS.doc)
FILE_TYPES = ["DEMO", "DRUG", "REAC", "OUTC", "RPSR", "THER", "INDI"]

In [ ]:
def discover_faers_files(base_dir: str, file_types: list) -> dict:
    """
    Recursively scans base_dir for FAERS quarterly text files and groups
    their paths by file type (DEMO, DRUG, REAC, OUTC, RPSR, THER, INDI).

    FAERS naming convention is <TYPE>yyQq.txt (e.g. DEMO25Q1.txt), but this
    matches case-insensitively and tolerates files buried in sub-folders,
    since manual downloads often preserve the FDA's original zip structure.
    Excludes STATyyQq files (Section C: informational null-count file, not
    a data file) and non-.txt files.
    """
    all_txt_files = glob.glob(os.path.join(base_dir, "**", "*.txt"), recursive=True) \
                   + glob.glob(os.path.join(base_dir, "**", "*.TXT"), recursive=True)

    grouped = {ftype: [] for ftype in file_types}
    for fpath in all_txt_files:
        fname = os.path.basename(fpath).upper()
        for ftype in file_types:
            # Matches e.g. DEMO25Q1.TXT but not STAT25Q1.TXT or DEMO_README.TXT
            if re.match(rf"^{ftype}\d{{2}}Q\d\.TXT$", fname):
                grouped[ftype].append(fpath)
                break

    # Sanity report: every type should have exactly 4 files (Q1-Q4)
    print("File discovery summary")
    print("-" * 40)
    for ftype, paths in grouped.items():
        status = "OK" if len(paths) == 4 else "CHECK — expected 4, found " + str(len(paths))
        print(f"{ftype:6s}: {len(paths)} file(s)  [{status}]")
        for p in sorted(paths):
            print(f"         {p}")
    return grouped

faers_files = discover_faers_files(BASE_DIR, FILE_TYPES)

File discovery summary
----------------------------------------
DEMO  : 4 file(s)  [OK]
         /content/drive/MyDrive/FAERS 2025/Q1/DEMO25Q1.txt
         /content/drive/MyDrive/FAERS 2025/Q2/DEMO25Q2.txt
         /content/drive/MyDrive/FAERS 2025/Q3/DEMO25Q3.txt
         /content/drive/MyDrive/FAERS 2025/Q4/DEMO25Q4.txt
DRUG  : 4 file(s)  [OK]
         /content/drive/MyDrive/FAERS 2025/Q1/DRUG25Q1.txt
         /content/drive/MyDrive/FAERS 2025/Q2/DRUG25Q2.txt
         /content/drive/MyDrive/FAERS 2025/Q3/DRUG25Q3.txt
         /content/drive/MyDrive/FAERS 2025/Q4/DRUG25Q4.txt
REAC  : 4 file(s)  [OK]
         /content/drive/MyDrive/FAERS 2025/Q1/REAC25Q1.txt
         /content/drive/MyDrive/FAERS 2025/Q2/REAC25Q2.txt
         /content/drive/MyDrive/FAERS 2025/Q3/REAC25Q3.txt
         /content/drive/MyDrive/FAERS 2025/Q4/REAC25Q4.txt
OUTC  : 4 file(s)  [OK]
         /content/drive/MyDrive/FAERS 2025/Q1/OUTC25Q1.txt
         /content/drive/MyDrive/FAERS 2025/Q2/OUTC25Q2.txt
         /cont

In [ ]:
def load_faers_file(filepath: str) -> pd.DataFrame:
    """
    Loads a single FAERS ASCII file exactly as delivered, with no cleaning.

    Design decisions (documented for the final report's methodology section):
      - sep='$'                : FAERS files are dollar-delimited (Section A).
      - dtype=str               : everything read as text at ingestion. Coercing
                                   to numeric/date types this early risks silent
                                   misparsing before any validity checks have run;
                                   real typing happens deliberately in the
                                   cleaning phase.
      - encoding='latin-1'      : FAERS extracts are known to contain non-UTF-8
                                   bytes (e.g. accented characters in drug names
                                   and literature references); UTF-8 raises
                                   decode errors on these files.
      - engine='python'         : more tolerant of the occasional malformed row
                                   than the default C engine.
      - on_bad_lines='warn'     : malformed rows are logged and skipped rather
                                   than silently dropped or crashing the load;
                                   the count feeds into data-quality reporting.
    """
    df = pd.read_csv(
        filepath,
        sep="$",
        dtype=str,
        encoding="latin-1",
        engine="python",
        on_bad_lines="warn",
        quoting=3,          # csv.QUOTE_NONE — FAERS text fields aren't quote-escaped
    )
    df.columns = [c.strip().lower() for c in df.columns]  # normalize header casing/whitespace only
    return df

def load_and_concatenate(file_type: str, filepaths: list) -> pd.DataFrame:
    """
    Loads all 4 quarterly files for one FAERS file type and stacks them into
    a single full-year dataframe, tagging each row with its source quarter
    (needed later for the per-quarter missingness breakdown in Phase 3/10).
    """
    quarter_frames = []
    for fpath in sorted(filepaths):
        quarter_label = re.search(r"(\d{2}Q\d)", os.path.basename(fpath).upper()).group(1)
        df = load_faers_file(fpath)
        df["source_quarter"] = quarter_label
        quarter_frames.append(df)
        print(f"  loaded {os.path.basename(fpath):20s} -> {df.shape[0]:>9,} rows, {df.shape[1]} cols")
    combined = pd.concat(quarter_frames, ignore_index=True)
    return combined

In [ ]:
raw_data = {}
for ftype in FILE_TYPES:
    print(f"\nLoading {ftype} (4 quarters)...")
    raw_data[ftype] = load_and_concatenate(ftype, faers_files[ftype])
    print(f"  -> {ftype} full-year combined shape: {raw_data[ftype].shape}")


Loading DEMO (4 quarters)...
  loaded DEMO25Q1.txt         ->   400,514 rows, 26 cols
  loaded DEMO25Q2.txt         ->   393,130 rows, 26 cols
  loaded DEMO25Q3.txt         ->   438,512 rows, 26 cols
  loaded DEMO25Q4.txt         ->   385,288 rows, 26 cols
  -> DEMO full-year combined shape: (1617444, 26)

Loading DRUG (4 quarters)...
  loaded DRUG25Q1.txt         -> 2,008,162 rows, 21 cols
  loaded DRUG25Q2.txt         -> 1,829,056 rows, 21 cols
  loaded DRUG25Q3.txt         -> 2,148,451 rows, 21 cols
  loaded DRUG25Q4.txt         -> 1,815,349 rows, 21 cols
  -> DRUG full-year combined shape: (7801018, 21)

Loading REAC (4 quarters)...
  loaded REAC25Q1.txt         -> 1,432,926 rows, 5 cols
  loaded REAC25Q2.txt         -> 1,340,666 rows, 5 cols
  loaded REAC25Q3.txt         -> 1,535,133 rows, 5 cols
  loaded REAC25Q4.txt         -> 1,349,105 rows, 5 cols
  -> REAC full-year combined shape: (5657830, 5)

Loading OUTC (4 quarters)...
  loaded OUTC25Q1.txt         ->   304,027 rows, 4 

In [ ]:
print("\n" + "=" * 60)
print("RAW INGESTION SUMMARY")
print("=" * 60)
for ftype, df in raw_data.items():
    n_rows = len(df)
    n_primaryid = df["primaryid"].nunique() if "primaryid" in df.columns else "n/a"
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{ftype:6s} | rows: {n_rows:>9,} | unique primaryid: {str(n_primaryid):>9s} "
          f"| memory: {mem_mb:6.1f} MB")

# DEMO should have exactly one row per primaryid (Section C.1: "a single
# record for each event report"). This is the first real data-quality check —
# flag it now, resolve/document it in Phase 3, don't drop anything yet.
demo_dupe_count = raw_data["DEMO"]["primaryid"].duplicated().sum()
print(f"\nDEMO duplicate primaryid rows: {demo_dupe_count} "
      f"({'OK — expected 0' if demo_dupe_count == 0 else 'FLAG for Phase 3 cleaning'})")


RAW INGESTION SUMMARY
DEMO   | rows: 1,617,444 | unique primaryid:   1617313 | memory: 2086.8 MB
DRUG   | rows: 7,801,018 | unique primaryid:   1617313 | memory: 7503.8 MB
REAC   | rows: 5,657,830 | unique primaryid:   1617313 | memory: 1504.3 MB
OUTC   | rows: 1,232,582 | unique primaryid:    919627 | memory:  270.0 MB
RPSR   | rows:    43,939 | unique primaryid:     43341 | memory:    9.6 MB
THER   | rows: 2,002,380 | unique primaryid:    792212 | memory:  782.7 MB
INDI   | rows: 4,821,589 | unique primaryid:   1504516 | memory: 1411.9 MB

DEMO duplicate primaryid rows: 131 (FLAG for Phase 3 cleaning)


In [ ]:
# Avoids re-parsing 28 raw text files every time the runtime restarts.
# These are the untouched raw pulls — later cleaning steps will load from
# here, clean, and save their own separate cleaned-data cache.
for ftype, df in raw_data.items():
    cache_path = os.path.join(CACHE_DIR, f"raw_{ftype.lower()}.pkl")
    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"Cached {ftype} -> {cache_path}")

print("\nStep 1 complete: 7 raw full-year dataframes loaded, validated, and cached.")

Cached DEMO -> /content/drive/MyDrive/FAERS 2025/_cache/raw_demo.pkl


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 2 — COLUMN AUDIT & TRIMMING
# ------------------------------------------------------------------------------
# Purpose : Audit null/blank rates on every column of all 7 raw file types
#           (this is the "column-by-column review" the project plan calls for,
#           and it's also the source data for the report's missingness
#           section in Phase 10). Then trim each file to only the columns
#           needed to answer the 13 questions, with the reasoning for every
#           drop documented inline rather than silently discarded.
# Input   : raw_*.pkl cache files written in Step 1.
# Output  : 7 trimmed dataframes, cached separately from the raw cache so the
#           original raw pulls stay untouched and re-runnable.
# ==============================================================================
import os
import pickle
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

FILE_TYPES = ["DEMO", "DRUG", "REAC", "OUTC", "RPSR", "THER", "INDI"]

raw_data = {}
for ftype in FILE_TYPES:
    with open(os.path.join(CACHE_DIR, f"raw_{ftype.lower()}.pkl"), "rb") as f:
        raw_data[ftype] = pickle.load(f)
    print(f"Reloaded {ftype}: {raw_data[ftype].shape}")

Reloaded DEMO: (1617444, 26)
Reloaded DRUG: (7801018, 21)
Reloaded REAC: (5657830, 5)
Reloaded OUTC: (1232582, 4)
Reloaded RPSR: (43939, 4)
Reloaded THER: (2002380, 8)
Reloaded INDI: (4821589, 5)


In [ ]:
def null_blank_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Reports, per column, the percentage of rows that are missing — where
    "missing" covers both true NaN and empty/whitespace-only strings, since
    FAERS $-delimited files commonly leave a field blank between delimiters
    rather than omitting it entirely (these load as '' with dtype=str, not
    NaN, so a plain .isnull() check alone would understate missingness).

    This is the evidence base for both the column-trimming decisions below
    and the per-field missingness section required in the final report
    (Phase 10) — it documents what's missing, not just what's kept.
    """
    n = len(df)
    rows = []
    for col in df.columns:
        if col == "source_quarter":
            continue
        is_missing = df[col].isna() | (df[col].astype(str).str.strip() == "")
        missing_pct = round(is_missing.sum() / n * 100, 2)
        rows.append({"column": col, "missing_pct": missing_pct,
                      "non_null_sample": df.loc[~is_missing, col].iloc[0] if (~is_missing).any() else None})
    report = pd.DataFrame(rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)
    print(f"\n{'=' * 60}\n{name} — column missingness audit ({n:,} rows)\n{'=' * 60}")
    print(report.to_string(index=False))
    return report

audit_reports = {}
for ftype, df in raw_data.items():
    audit_reports[ftype] = null_blank_report(df, ftype)


DEMO — column missingness audit (1,617,444 rows)
          column  missing_pct                                                                                                                                                                                           non_null_sample
          to_mfr        96.81                                                                                                                                                                                                         N
         lit_ref        90.86 Bianchi G V, Duca M, Sica L, Mariani G.. Metastatic breast cancer treated with lapatinib with a prolonged benefit: A case report and a review of therapeutic options available. TUMORI. 2013;99(6):269-72
        auth_num        90.76                                                                                                                                                                                     CA-JNJFOC-20150420970
              wt      

In [ ]:
# Each list below maps directly to which of the 13 questions need the field.
# Columns NOT listed are dropped from the working dataset at this stage —
# note this drops the COLUMN, not any rows; no records are removed here.

KEEP_COLUMNS = {

    "DEMO": [
        "primaryid", "caseid", "caseversion", "i_f_code",   # keys + version info,
                                                              # needed in Phase 3 to
                                                              # resolve the 131 duplicate
                                                              # primaryid rows flagged
                                                              # in Step 1 (keep latest
                                                              # case version per caseid)
        "event_dt", "mfr_dt", "init_fda_dt", "fda_dt",       # Q2, Q3: reporting lag calcs
        "age", "age_cod",                                    # Q1: age distribution
        "age_grp",                                           # Q1: pre-coded age bucket, cross-check against derived age_cod bucket
        "sex",                                               # Q1: gender distribution
        "reporter_country", "occr_country",                  # Q4, Q10: country volume/death-rate
                                                              # (kept both — reporter_country
                                                              # = who submitted, occr_country
                                                              # = where the event happened;
                                                              # decided which one answers
                                                              # Q4/Q10 during that analysis)
        "occp_cod",                                          # supplementary cross-check for Q11 (reporter type vs RPSR source)
    ],
    # DROPPED: rept_cod (report timing category — not asked by any question,
    #   and explicitly flagged for exclusion in the original plan), auth_num
    #   and mfr_num (regulatory/manufacturer case IDs — no analytical use),
    #   mfr_sndr (manufacturer name — not a question dimension), lit_ref
    #   (free-text literature citation — not structured data), e_sub (electronic
    #   submission flag — not a question dimension), wt/wt_cod (weight — no
    #   question uses it; also explicitly flagged for exclusion in the plan),
    #   rept_dt (date report was sent — distinct from fda_dt/mfr_dt, not used
    #   by any lag question), to_mfr (voluntary-reporter flag — not used).

    "DRUG": [
        "primaryid", "caseid", "drug_seq",                   # keys (drug_seq links to THER/INDI)
        "role_cod",                                          # needed to isolate Primary Suspect (PS) drugs for Q5-Q8, Q12-Q13
        "drugname",                                          # fallback display name when prod_ai is blank
        "prod_ai",                                           # Q5-Q8, Q12, Q13: primary drug identifier by active ingredient
        "dechal", "rechal",                                  # Q13: dechallenge/rechallenge causality evidence
    ],
    # DROPPED: val_vbm (source-of-name flag — superseded by prod_ai as the
    #   analysis key), route/dose_vbm/dose_amt/dose_unit/dose_form/dose_freq
    #   (dosing detail — none of the 13 questions analyze dose or route),
    #   cum_dose_chr/cum_dose_unit (cumulative dose — same reasoning), lot_num
    #   and exp_dt (manufacturing/QA fields, not adverse-event analytics),
    #   nda_num (regulatory approval number — not a question dimension).

    "REAC": [
        "primaryid", "caseid", "pt",                          # Q7, Q9, Q13: the reaction term itself
    ],
    # DROPPED: drug_rec_act — explicitly flagged for exclusion in the plan;
    #   it's a free-text re-statement of the PT on positive rechallenge, and
    #   RECHAL in the DRUG file already captures that signal structurally.

    "OUTC": [
        "primaryid", "caseid", "outc_cod",                   # Q6, Q9, Q10, Q11, Q12: every outcome/severity question
    ],
    # No drops — file is already minimal and every field is used.

    "RPSR": [
        "primaryid", "caseid", "rpsr_cod",                   # Q11: reporter source vs. severity
    ],
    # No drops — file is already minimal and every field is used.

    "THER": [
        "primaryid", "caseid", "dsg_drug_seq",               # keys (dsg_drug_seq links back to DRUG.drug_seq)
        "start_dt", "end_dt",                                # Q12: therapy duration, calculated from dates where possible
        "dur", "dur_cod",                                    # Q12: fallback duration when start/end dates are incomplete
    ],
    # No drops — file is already minimal and every field is used.

    "INDI": [
        "primaryid", "caseid", "indi_drug_seq", "indi_pt",   # Q8: indication for top drugs (indi_drug_seq links to DRUG.drug_seq)
    ],
    # No drops — file is already minimal and every field is used.
}

In [ ]:
trimmed_data = {}
for ftype, df in raw_data.items():
    keep = KEEP_COLUMNS[ftype] + ["source_quarter"]
    dropped = [c for c in df.columns if c not in keep]
    trimmed_data[ftype] = df[keep].copy()
    print(f"{ftype:6s} | kept {len(keep)-1} cols + source_quarter | dropped {len(dropped)}: {dropped}")

print("\nFinal trimmed shapes:")
for ftype, df in trimmed_data.items():
    print(f"  {ftype:6s}: {df.shape}")

DEMO   | kept 15 cols + source_quarter | dropped 10: ['rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr', 'lit_ref', 'e_sub', 'wt', 'wt_cod', 'rept_dt', 'to_mfr']
DRUG   | kept 8 cols + source_quarter | dropped 12: ['val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq']
REAC   | kept 3 cols + source_quarter | dropped 1: ['drug_rec_act']
OUTC   | kept 3 cols + source_quarter | dropped 0: []
RPSR   | kept 3 cols + source_quarter | dropped 0: []
THER   | kept 7 cols + source_quarter | dropped 0: []
INDI   | kept 4 cols + source_quarter | dropped 0: []

Final trimmed shapes:
  DEMO  : (1617444, 16)
  DRUG  : (7801018, 9)
  REAC  : (5657830, 4)
  OUTC  : (1232582, 4)
  RPSR  : (43939, 4)
  THER  : (2002380, 8)
  INDI  : (4821589, 5)


In [ ]:
# The plan calls this out separately since these three files are already
# narrow — this confirms their retained columns carry real analytical signal
# rather than being mostly blank/uninformative, before we finalize them.
print("\nOUTC — outc_cod value distribution:")
print(trimmed_data["OUTC"]["outc_cod"].value_counts(dropna=False))

print("\nRPSR — rpsr_cod value distribution:")
print(trimmed_data["RPSR"]["rpsr_cod"].value_counts(dropna=False))

print("\nINDI — indi_pt top 15 most frequent indications:")
print(trimmed_data["INDI"]["indi_pt"].value_counts(dropna=False).head(15))

print("\nINDI — % rows with blank indi_pt:",
      round((trimmed_data["INDI"]["indi_pt"].isna() |
             (trimmed_data["INDI"]["indi_pt"].astype(str).str.strip() == "")).mean() * 100, 2), "%")


OUTC — outc_cod value distribution:
outc_cod
OT    682845
HO    336991
DE    121439
LT     54308
DS     25930
CA      6082
RI      4987
Name: count, dtype: int64

RPSR — rpsr_cod value distribution:
rpsr_cod
CSM    24309
HP     19161
FGN      469
Name: count, dtype: int64

INDI — indi_pt top 15 most frequent indications:
indi_pt
Product used for unknown indication    1921202
Rheumatoid arthritis                    265491
Dermatitis atopic                        76103
Type 2 diabetes mellitus                 60193
Asthma                                   59801
Weight control                           57478
Crohn's disease                          52596
Plasma cell myeloma                      50526
Hypertension                             46800
Psoriasis                                41193
Colitis ulcerative                       41069
Migraine                                 36370
Prophylaxis                              34857
Diabetes mellitus                        33687
Psoriatic 

In [ ]:
for ftype, df in trimmed_data.items():
    cache_path = os.path.join(CACHE_DIR, f"trimmed_{ftype.lower()}.pkl")
    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"Cached trimmed {ftype} -> {cache_path}")

print("\nStep 2 complete: columns audited and trimmed for all 7 file types.")

Cached trimmed DEMO -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_demo.pkl
Cached trimmed DRUG -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_drug.pkl
Cached trimmed REAC -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_reac.pkl
Cached trimmed OUTC -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_outc.pkl
Cached trimmed RPSR -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_rpsr.pkl
Cached trimmed THER -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_ther.pkl
Cached trimmed INDI -> /content/drive/MyDrive/FAERS 2025/_cache/trimmed_indi.pkl

Step 2 complete: columns audited and trimmed for all 7 file types.


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 3 — CLEANING: DEMOGRAPHIC (DEMO) FILE
# ------------------------------------------------------------------------------
# Purpose : Resolve the 131 duplicate primaryid rows flagged in Step 1, parse
#           and validate the three date fields needed for Q2/Q3, and derive a
#           normalized, plausibility-checked age in years for Q1.
# Input   : trimmed_demo.pkl from Step 2.
# Output  : clean_demo.pkl — one row per report, with event_dt_clean,
#           mfr_dt_clean, fda_dt_clean, and age_years columns added.
# Principle: nothing is silently dropped. Implausible values are set to NaN
#           (not deleted as rows) and every correction is counted and printed
#           so it can go straight into the report's data-quality section.
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

with open(os.path.join(CACHE_DIR, "trimmed_demo.pkl"), "rb") as f:
    demo = pickle.load(f)

print(f"Loaded trimmed DEMO: {demo.shape}")

Loaded trimmed DEMO: (1617444, 16)


In [ ]:
# primaryid is defined (Section D) as the unique key for a report, so any
# duplicate is a data artifact, not new information. Two possible causes:
#   (a) exact full-row duplicates (e.g. a case republished unchanged across
#       adjacent quarterly extracts)
#   (b) same primaryid with DIFFERING field values across the duplicate rows
#       (would indicate an actual data conflict, not just redundancy)
# These are handled differently and the split is reported, not assumed.

dupe_mask = demo["primaryid"].duplicated(keep=False)
dupes = demo[dupe_mask].sort_values("primaryid")
print(f"Total rows involved in primaryid duplication: {len(dupes)} "
      f"({dupes['primaryid'].nunique()} distinct primaryid values)")

# Check, per duplicated primaryid, whether all non-key columns actually match
compare_cols = [c for c in demo.columns if c not in ("source_quarter",)]
def rows_fully_match(group):
    return group[compare_cols].drop_duplicates().shape[0] == 1

fully_matching = dupes.groupby("primaryid").apply(rows_fully_match)
n_exact = fully_matching.sum()
n_conflicting = (~fully_matching).sum()
print(f"  -> {n_exact} primaryid groups are exact duplicates (safe to collapse)")
print(f"  -> {n_conflicting} primaryid groups have CONFLICTING data across duplicate rows")

if n_conflicting > 0:
    conflicting_ids = fully_matching[~fully_matching].index.tolist()
    print(f"  Conflicting primaryid values (inspect manually if needed): {conflicting_ids[:20]}")

# Resolution: for exact duplicates, keep one copy. For conflicting duplicates,
# keep the row from the LATEST source_quarter — FAERS extracts are cumulative
# snapshots, so a later quarter's copy of the same case reflects the most
# recent manufacturer/FDA update.
demo_sorted = demo.sort_values("source_quarter")
demo_dedup = demo_sorted.drop_duplicates(subset="primaryid", keep="last").reset_index(drop=True)

rows_removed = len(demo) - len(demo_dedup)
print(f"\nDEMO rows before dedup: {len(demo):,} | after dedup: {len(demo_dedup):,} "
      f"| rows removed: {rows_removed} (all were confirmed redundant/superseded copies of an existing primaryid, no unique reports lost)")

demo = demo_dedup

Total rows involved in primaryid duplication: 262 (131 distinct primaryid values)


/tmp/ipykernel_4145/125345297.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fully_matching = dupes.groupby("primaryid").apply(rows_fully_match)


  -> 131 primaryid groups are exact duplicates (safe to collapse)
  -> 0 primaryid groups have CONFLICTING data across duplicate rows

DEMO rows before dedup: 1,617,444 | after dedup: 1,617,313 | rows removed: 131 (all were confirmed redundant/superseded copies of an existing primaryid, no unique reports lost)


In [ ]:
def parse_faers_date(series: pd.Series) -> tuple:
    """
    Parses FAERS date fields, which per the ASC_NTS NOTE can arrive as full
    dates (YYYYMMDD), year-month (YYYYMM), or year-only (YYYY).

    Returns:
      parsed   : datetime64 series (day/month defaulted to 01 when the source
                 date was partial — the granularity column below records this
                 so downstream lag calculations can account for reduced
                 precision rather than silently treating a partial date as
                 exact).
      granularity : 'day' / 'month' / 'year' / 'invalid' / 'missing' per row.
    """
    s = series.astype(str).str.strip()
    s = s.replace({"nan": "", "None": ""})

    granularity = pd.Series(index=s.index, dtype=object)
    granularity[s == ""] = "missing"
    granularity[s.str.len() == 8] = "day"
    granularity[s.str.len() == 6] = "month"
    granularity[s.str.len() == 4] = "year"
    granularity[granularity.isna()] = "invalid"   # any other length is malformed

    # Pad month/year-only values to a full date string for parsing, using the
    # 1st of the month / 1st of January as the anchor day — this is a
    # standard, documented convention for partial-date imputation, not a
    # silent guess (the granularity column preserves that it was partial).
    padded = s.copy()
    padded[granularity == "month"] = s[granularity == "month"] + "01"
    padded[granularity == "year"] = s[granularity == "year"] + "0101"
    padded[granularity.isin(["missing", "invalid"])] = np.nan

    parsed = pd.to_datetime(padded, format="%Y%m%d", errors="coerce")

    # Any row that had a length-8/6/4 string but STILL failed to parse
    # (e.g. month 13, day 32) is a genuine invalid date, not just partial.
    newly_invalid = parsed.isna() & granularity.isin(["day", "month", "year"])
    granularity[newly_invalid] = "invalid"

    return parsed, granularity

for col in ["event_dt", "mfr_dt", "init_fda_dt", "fda_dt"]:
    parsed, gran = parse_faers_date(demo[col])
    demo[f"{col}_clean"] = parsed
    demo[f"{col}_granularity"] = gran
    print(f"\n{col} granularity breakdown:")
    print(gran.value_counts())


event_dt granularity breakdown:
missing    910036
day        707272
invalid         5
Name: count, dtype: int64

mfr_dt granularity breakdown:
day        1617312
invalid          1
Name: count, dtype: int64

init_fda_dt granularity breakdown:
day    1617313
Name: count, dtype: int64

fda_dt granularity breakdown:
day    1617313
Name: count, dtype: int64


In [ ]:
# Range: 1985-2026. FAERS itself dates back to 1969, so this threshold is a
# deliberate, documented judgment call (not a hard system limit) to catch
# clear data-entry typos (e.g. a stray "1905" or "2099") while still covering
# any legitimately old event dates likely to appear in a 2025 extract.
MIN_YEAR, MAX_YEAR = 1985, 2026

date_cols_clean = ["event_dt_clean", "mfr_dt_clean", "init_fda_dt_clean", "fda_dt_clean"]
for col in date_cols_clean:
    out_of_range = demo[col].dt.year.notna() & ((demo[col].dt.year < MIN_YEAR) | (demo[col].dt.year > MAX_YEAR))
    n_flagged = out_of_range.sum()
    print(f"{col}: {n_flagged} values outside {MIN_YEAR}-{MAX_YEAR}, set to NaT")
    demo.loc[out_of_range, col] = pd.NaT
    # keep the granularity label as 'invalid' for these, for accurate reporting
    gran_col = col.replace("_clean", "_granularity")
    if gran_col in demo.columns:
        demo.loc[out_of_range, gran_col] = "invalid_out_of_range"

event_dt_clean: 218 values outside 1985-2026, set to NaT
mfr_dt_clean: 2 values outside 1985-2026, set to NaT
init_fda_dt_clean: 0 values outside 1985-2026, set to NaT
fda_dt_clean: 0 values outside 1985-2026, set to NaT


In [ ]:
AGE_UNIT_TO_YEARS = {
    "DEC": 10.0,
    "YR": 1.0,
    "MON": 1 / 12,
    "WK": 1 / 52.1429,
    "DY": 1 / 365.25,
    "HR": 1 / 8760,
}

age_numeric = pd.to_numeric(demo["age"], errors="coerce")
age_cod_clean = demo["age_cod"].astype(str).str.strip().str.upper()

unrecognized_unit = age_numeric.notna() & ~age_cod_clean.isin(AGE_UNIT_TO_YEARS.keys())
print(f"Rows with a numeric age but an unrecognized/missing age_cod: {unrecognized_unit.sum()} "
      f"(age_years left as NaN for these — can't convert without a known unit)")

conversion_factor = age_cod_clean.map(AGE_UNIT_TO_YEARS)
demo["age_years"] = age_numeric * conversion_factor

Rows with a numeric age but an unrecognized/missing age_cod: 1 (age_years left as NaN for these — can't convert without a known unit)


In [ ]:
# Range: 0-122 years. 122 is the oldest medically verified human lifespan on
# record (Jeanne Calment), so this is a genuine biological ceiling, not an
# arbitrary cutoff.
implausible_age = demo["age_years"].notna() & ((demo["age_years"] < 0) | (demo["age_years"] > 122))
n_implausible = implausible_age.sum()
print(f"\nImplausible age_years values (outside 0-122): {n_implausible} "
      f"({round(n_implausible / demo['age_years'].notna().sum() * 100, 3)}% of non-null ages), set to NaN")
demo.loc[implausible_age, "age_years"] = np.nan

print(f"\nFinal age_years missingness: "
      f"{round(demo['age_years'].isna().mean() * 100, 2)}% "
      f"(includes originally-missing age, unrecognized units, and implausible values)")


Implausible age_years values (outside 0-122): 31 (0.003% of non-null ages), set to NaN

Final age_years missingness: 39.69% (includes originally-missing age, unrecognized units, and implausible values)


In [ ]:
# Confirms SEX only takes the 3 documented codes (Section D: UNK/M/F); any
# stray/blank value is standardized to 'UNK' rather than left as a silent
# fourth category.
print("\nsex value counts before standardization:")
print(demo["sex"].value_counts(dropna=False))

valid_sex = {"M", "F", "UNK"}
demo["sex"] = demo["sex"].astype(str).str.strip().str.upper()
demo.loc[~demo["sex"].isin(valid_sex), "sex"] = "UNK"

print("\nsex value counts after standardization:")
print(demo["sex"].value_counts(dropna=False))

# --- CELL 8: Summary & cache ---
print("\n" + "=" * 60)
print("DEMO CLEANING SUMMARY")
print("=" * 60)
print(f"Final row count               : {len(demo):,}")
print(f"Duplicate primaryid rows removed : {rows_removed}")
for col in date_cols_clean:
    print(f"{col:20s} missing/invalid: {round(demo[col].isna().mean() * 100, 2)}%")
print(f"age_years            missing/invalid: {round(demo['age_years'].isna().mean() * 100, 2)}%")

cache_path = os.path.join(CACHE_DIR, "clean_demo.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(demo, f)
print(f"\nCached cleaned DEMO -> {cache_path}")
print("Step 3 (DEMO cleaning) complete.")


sex value counts before standardization:
sex
F      775986
M      520398
NaN    319893
UNK      1036
Name: count, dtype: int64

sex value counts after standardization:
sex
F      775986
M      520398
UNK    320929
Name: count, dtype: int64

DEMO CLEANING SUMMARY
Final row count               : 1,617,313
Duplicate primaryid rows removed : 131
event_dt_clean       missing/invalid: 56.28%
mfr_dt_clean         missing/invalid: 0.0%
init_fda_dt_clean    missing/invalid: 0.0%
fda_dt_clean         missing/invalid: 0.0%
age_years            missing/invalid: 39.69%

Cached cleaned DEMO -> /content/drive/MyDrive/FAERS 2025/_cache/clean_demo.pkl
Step 3 (DEMO cleaning) complete.


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 4 — CLEANING: DRUG FILE
# ------------------------------------------------------------------------------
# Purpose : Resolve any cross-quarter duplicate (primaryid, drug_seq) rows,
#           validate role_cod against the documented code set, normalize
#           prod_ai/drugname text so grouping/Pareto analysis isn't split by
#           casing or whitespace, and standardize dechal/rechal into a clean
#           categorical field that distinguishes "not reported" from the
#           documented "Unknown" code.
# Input   : trimmed_drug.pkl from Step 2.
# Output  : clean_drug.pkl
# Principle: same as Step 3 — nothing dropped without a documented, counted
#           reason; implausible/unexpected values are flagged, not deleted.
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

with open(os.path.join(CACHE_DIR, "trimmed_drug.pkl"), "rb") as f:
    drug = pickle.load(f)

print(f"Loaded trimmed DRUG: {drug.shape}")

Loaded trimmed DRUG: (7801018, 9)


In [ ]:
# drug_seq only uniquely identifies a drug WITHIN a given primaryid (Section
# D: "Unique number for identifying a drug for a Case"), so the natural key
# here is the (primaryid, drug_seq) pair, not drug_seq alone. Same redundancy
# risk as DEMO's primaryid duplication: a case can appear unchanged across
# adjacent quarterly extracts.
key_cols = ["primaryid", "drug_seq"]
dupe_mask = drug.duplicated(subset=key_cols, keep=False)
dupes = drug[dupe_mask]
n_dupe_rows = len(dupes)
n_dupe_keys = dupes[key_cols].drop_duplicates().shape[0]
print(f"Rows involved in (primaryid, drug_seq) duplication: {n_dupe_rows:,} "
      f"({n_dupe_keys:,} distinct keys)")

if n_dupe_rows > 0:
    compare_cols = [c for c in drug.columns if c not in ("source_quarter",)]
    fully_matching = dupes.groupby(key_cols)[compare_cols].apply(
        lambda g: g.drop_duplicates().shape[0] == 1
    )
    n_exact = fully_matching.sum()
    n_conflicting = (~fully_matching).sum()
    print(f"  -> {n_exact} keys are exact duplicates (safe to collapse)")
    print(f"  -> {n_conflicting} keys have CONFLICTING data across duplicate rows")

    drug_sorted = drug.sort_values("source_quarter")
    drug_dedup = drug_sorted.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
    rows_removed = len(drug) - len(drug_dedup)
    drug = drug_dedup
    print(f"DRUG rows before dedup: {n_dupe_rows + (len(drug) - n_dupe_keys):,} "
          f"| rows removed: {rows_removed}")
else:
    rows_removed = 0
    print("No duplicates found — no rows removed.")

print(f"DRUG shape after dedup: {drug.shape}")

Rows involved in (primaryid, drug_seq) duplication: 491,580 (160,381 distinct keys)
  -> 160381 keys are exact duplicates (safe to collapse)
  -> 0 keys have CONFLICTING data across duplicate rows
DRUG rows before dedup: 7,801,018 | rows removed: 331199
DRUG shape after dedup: (7469819, 9)


In [ ]:
# Documented codes (Section D, updated Jan 2025): PS, SS, C, I, DN.
VALID_ROLE_COD = {"PS", "SS", "C", "I", "DN"}
print("\nrole_cod value counts:")
print(drug["role_cod"].value_counts(dropna=False))

unexpected_role = ~drug["role_cod"].astype(str).str.strip().str.upper().isin(VALID_ROLE_COD)
print(f"\nRows with an unexpected role_cod value: {unexpected_role.sum()} "
      f"(left as-is, flagged for report — not dropped)")


role_cod value counts:
role_cod
SS    2986321
C     2828901
PS    1617303
I       37137
DN        157
Name: count, dtype: int64

Rows with an unexpected role_cod value: 0 (left as-is, flagged for report — not dropped)


In [ ]:
# Strip whitespace and uppercase both fields so grouping/Pareto analysis in
# Q5-Q8 isn't fragmented by casing or trailing-space variants of the same
# active ingredient.
def normalize_text(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.upper()
    s = s.replace({"NAN": np.nan, "": np.nan})
    return s

drug["prod_ai_clean"] = normalize_text(drug["prod_ai"])
drug["drugname_clean"] = normalize_text(drug["drugname"])

# Fallback: where prod_ai is blank, use the normalized drugname instead, and
# track WHICH source each row's final label came from — this matters because
# drugname is verbatim/trade-name text (Section D.2), so it's noisier than
# prod_ai and any drug-level ranking built partly from it should be able to
# footnote how much of it rests on the fallback.
drug["drug_label"] = drug["prod_ai_clean"].fillna(drug["drugname_clean"])
drug["drug_label_source"] = np.select(
    [drug["prod_ai_clean"].notna(), drug["prod_ai_clean"].isna() & drug["drugname_clean"].notna()],
    ["prod_ai", "drugname_fallback"],
    default="missing",
)

print("\ndrug_label_source breakdown:")
print(drug["drug_label_source"].value_counts())
print(f"\n{round((drug['drug_label_source'] == 'drugname_fallback').mean() * 100, 2)}% "
      f"of rows rely on the drugname fallback instead of a reported prod_ai")


drug_label_source breakdown:
drug_label_source
prod_ai              7337721
drugname_fallback     132098
Name: count, dtype: int64

1.77% of rows rely on the drugname fallback instead of a reported prod_ai


In [ ]:
# Documented codes (Section D.2): Y (positive), N (negative), U (unknown),
# D (does not apply). A BLANK field is functionally different from 'U' —
# blank means the field wasn't populated at all, 'U' means it was actively
# reported as unknown — so these are kept as distinct categories rather than
# merged, to avoid overstating how much "unknown" causality evidence exists.
VALID_DECHAL_RECHAL = {"Y", "N", "U", "D"}

for col in ["dechal", "rechal"]:
    raw = drug[col].astype(str).str.strip().str.upper()
    is_blank = raw.isin(["", "NAN"])
    unexpected = ~raw.isin(VALID_DECHAL_RECHAL) & ~is_blank

    clean_col = f"{col}_clean"
    drug[clean_col] = raw
    drug.loc[is_blank, clean_col] = "NOT_REPORTED"     # explicit category, distinct from 'U'
    drug.loc[unexpected, clean_col] = "NOT_REPORTED"   # any stray/unrecognized value also folded here, count reported below

    print(f"\n{col} — value distribution after standardization:")
    print(drug[clean_col].value_counts(dropna=False))
    print(f"  blank/unreported: {is_blank.sum():,} | unexpected values folded in: {unexpected.sum():,}")


dechal — value distribution after standardization:
dechal_clean
NOT_REPORTED    3034902
U               2866471
D               1052857
Y                426116
N                 89473
Name: count, dtype: int64
  blank/unreported: 3,034,902 | unexpected values folded in: 0

rechal — value distribution after standardization:
rechal_clean
NOT_REPORTED    6774553
U                629575
N                 32647
D                 20350
Y                 12694
Name: count, dtype: int64
  blank/unreported: 6,774,553 | unexpected values folded in: 0


In [ ]:
print("\n" + "=" * 60)
print("DRUG CLEANING SUMMARY")
print("=" * 60)
print(f"Final row count                    : {len(drug):,}")
print(f"Duplicate (primaryid,drug_seq) rows removed : {rows_removed}")
print(f"role_cod unexpected values          : {unexpected_role.sum()}")
print(f"prod_ai fallback-to-drugname rate   : {round((drug['drug_label_source'] == 'drugname_fallback').mean() * 100, 2)}%")
print(f"drug_label still missing            : {(drug['drug_label_source'] == 'missing').sum()}")

cache_path = os.path.join(CACHE_DIR, "clean_drug.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(drug, f)
print(f"\nCached cleaned DRUG -> {cache_path}")
print("Step 4 (DRUG cleaning) complete.")


DRUG CLEANING SUMMARY
Final row count                    : 7,469,819
Duplicate (primaryid,drug_seq) rows removed : 331199
role_cod unexpected values          : 0
prod_ai fallback-to-drugname rate   : 1.77%
drug_label still missing            : 0

Cached cleaned DRUG -> /content/drive/MyDrive/FAERS 2025/_cache/clean_drug.pkl
Step 4 (DRUG cleaning) complete.


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 5 — CLEANING: THERAPY (THER) FILE
# ------------------------------------------------------------------------------
# Purpose : Remove only exact full-row duplicates (NOT key-based dedup — see
#           note below), validate start_dt/end_dt, compute therapy duration
#           preferring the two reported dates, falling back to dur/dur_cod,
#           deriving a missing single date from the other date + duration
#           where possible, and applying a plausibility floor.
# Input   : trimmed_ther.pkl from Step 2.
# Output  : clean_ther.pkl
#
# IMPORTANT: unlike DEMO/DRUG, (primaryid, dsg_drug_seq) is NOT a safe
# dedup key here. ASC_NTS.doc Section F, End Note 2 gives a real documented
# example (case 30781401, Aricept) where the same drug was stopped and later
# restarted, producing two legitimate THER rows sharing the same primaryid
# and drug_seq but different start_dt/end_dt. Deduplicating on the key alone
# would silently delete a real second treatment course. Only rows that are
# identical across EVERY column are true redundant duplicates.
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

with open(os.path.join(CACHE_DIR, "trimmed_ther.pkl"), "rb") as f:
    ther = pickle.load(f)

print(f"Loaded trimmed THER: {ther.shape}")

Loaded trimmed THER: (2002380, 8)


In [ ]:
compare_cols = [c for c in ther.columns if c != "source_quarter"]
exact_dupe_mask = ther.duplicated(subset=compare_cols, keep="first")
n_exact_dupes = exact_dupe_mask.sum()
print(f"Exact full-row duplicates found: {n_exact_dupes:,} — removing these only "
      f"(key-only matches like the documented Aricept stop/restart case are preserved)")

ther = ther[~exact_dupe_mask].reset_index(drop=True)
print(f"THER shape after exact-duplicate removal: {ther.shape}")

Exact full-row duplicates found: 45,700 — removing these only (key-only matches like the documented Aricept stop/restart case are preserved)
THER shape after exact-duplicate removal: (1956680, 8)


In [ ]:
def parse_faers_date(series: pd.Series) -> tuple:
    """Parses FAERS YYYYMMDD/YYYYMM/YYYY dates; see Step 3 for full rationale."""
    s = series.astype(str).str.strip()
    s = s.replace({"nan": "", "None": ""})

    granularity = pd.Series(index=s.index, dtype=object)
    granularity[s == ""] = "missing"
    granularity[s.str.len() == 8] = "day"
    granularity[s.str.len() == 6] = "month"
    granularity[s.str.len() == 4] = "year"
    granularity[granularity.isna()] = "invalid"

    padded = s.copy()
    padded[granularity == "month"] = s[granularity == "month"] + "01"
    padded[granularity == "year"] = s[granularity == "year"] + "0101"
    padded[granularity.isin(["missing", "invalid"])] = np.nan

    parsed = pd.to_datetime(padded, format="%Y%m%d", errors="coerce")
    newly_invalid = parsed.isna() & granularity.isin(["day", "month", "year"])
    granularity[newly_invalid] = "invalid"
    return parsed, granularity

for col in ["start_dt", "end_dt"]:
    parsed, gran = parse_faers_date(ther[col])
    ther[f"{col}_clean"] = parsed
    ther[f"{col}_granularity"] = gran
    print(f"\n{col} granularity breakdown:")
    print(gran.value_counts())

# Same 1985-2026 plausibility range used for DEMO, for consistency across the
# whole project.
MIN_YEAR, MAX_YEAR = 1985, 2026
for col in ["start_dt_clean", "end_dt_clean"]:
    out_of_range = ther[col].dt.year.notna() & ((ther[col].dt.year < MIN_YEAR) | (ther[col].dt.year > MAX_YEAR))
    print(f"{col}: {out_of_range.sum()} values outside {MIN_YEAR}-{MAX_YEAR}, set to NaT")
    ther.loc[out_of_range, col] = pd.NaT


start_dt granularity breakdown:
day        1307327
month       328743
missing     167971
year        152630
invalid          9
Name: count, dtype: int64

end_dt granularity breakdown:
missing    1179896
day         624540
month        99227
year         53015
invalid          2
Name: count, dtype: int64
start_dt_clean: 892 values outside 1985-2026, set to NaT
end_dt_clean: 58 values outside 1985-2026, set to NaT


In [ ]:
DUR_UNIT_TO_DAYS = {
    "YR": 365.25,
    "MON": 30.44,   # average month length — standard convention for unit conversion, not a per-case estimate
    "WK": 7.0,
    "DAY": 1.0,
    "HR": 1 / 24,
    "MIN": 1 / 1440,
    "SEC": 1 / 86400,
}

dur_numeric = pd.to_numeric(ther["dur"], errors="coerce")
dur_cod_clean = ther["dur_cod"].astype(str).str.strip().str.upper()
unrecognized_dur_unit = dur_numeric.notna() & ~dur_cod_clean.isin(DUR_UNIT_TO_DAYS.keys())
print(f"\nRows with numeric dur but unrecognized/missing dur_cod: {unrecognized_dur_unit.sum()} "
      f"(dur_days left NaN for these)")

ther["dur_days"] = dur_numeric * dur_cod_clean.map(DUR_UNIT_TO_DAYS)


Rows with numeric dur but unrecognized/missing dur_cod: 61 (dur_days left NaN for these)


In [ ]:
# Only applied when exactly ONE of start_dt/end_dt is present and dur_days is
# available — this is an explicit, documented imputation, tracked via
# *_is_derived flags so it's never confused with an originally-reported date.
has_start = ther["start_dt_clean"].notna()
has_end = ther["end_dt_clean"].notna()
has_dur = ther["dur_days"].notna()

derive_end = has_start & ~has_end & has_dur
ther.loc[derive_end, "end_dt_clean"] = (
    ther.loc[derive_end, "start_dt_clean"] + pd.to_timedelta(ther.loc[derive_end, "dur_days"], unit="D")
)
print(f"\nend_dt derived from start_dt + dur_days: {derive_end.sum():,} rows")

derive_start = has_end & ~has_start & has_dur
ther.loc[derive_start, "start_dt_clean"] = (
    ther.loc[derive_start, "end_dt_clean"] - pd.to_timedelta(ther.loc[derive_start, "dur_days"], unit="D")
)
print(f"start_dt derived from end_dt - dur_days: {derive_start.sum():,} rows")

ther["start_dt_is_derived"] = derive_start
ther["end_dt_is_derived"] = derive_end


end_dt derived from start_dt + dur_days: 9,251 rows
start_dt derived from end_dt - dur_days: 2,062 rows


In [ ]:
both_dates_now = ther["start_dt_clean"].notna() & ther["end_dt_clean"].notna()
duration_from_dates = (ther["end_dt_clean"] - ther["start_dt_clean"]).dt.days

# Implausible: end before start. Flagged and set to NaN — not deleted.
negative_duration = both_dates_now & (duration_from_dates < 0)
print(f"\nRows where end_dt < start_dt (implausible): {negative_duration.sum():,}, duration set to NaN for these")
duration_from_dates[negative_duration] = np.nan

ther["duration_days"] = duration_from_dates
# Last-resort fallback: both dates still missing, but dur/dur_cod was usable directly
still_missing = ther["duration_days"].isna() & ther["dur_days"].notna()
ther.loc[still_missing, "duration_days"] = ther.loc[still_missing, "dur_days"]

ther["duration_source"] = np.select(
    [
        both_dates_now & ~negative_duration & ~ther["start_dt_is_derived"] & ~ther["end_dt_is_derived"],
        ther["start_dt_is_derived"] | ther["end_dt_is_derived"],
        still_missing,
    ],
    ["from_both_reported_dates", "from_derived_date", "from_dur_cod_only"],
    default="unavailable",
)
print("\nduration_source breakdown:")
print(ther["duration_source"].value_counts())


Rows where end_dt < start_dt (implausible): 18,205, duration set to NaN for these

duration_source breakdown:
duration_source
unavailable                 1146173
from_both_reported_dates     695057
from_dur_cod_only            104137
from_derived_date             11313
Name: count, dtype: int64


In [ ]:
below_floor = ther["duration_days"].notna() & (ther["duration_days"] < 1)
print(f"\nDurations below the 1-day plausibility floor: {below_floor.sum():,} "
      f"({round(below_floor.sum() / ther['duration_days'].notna().sum() * 100, 2)}% of computed durations), set to NaN")
ther.loc[below_floor, "duration_days"] = np.nan
ther.loc[below_floor, "duration_source"] = "unavailable"

# --- CELL 8: Summary & cache ---
print("\n" + "=" * 60)
print("THER CLEANING SUMMARY")
print("=" * 60)
print(f"Final row count                     : {len(ther):,}")
print(f"Exact full-row duplicates removed    : {n_exact_dupes}")
print(f"Final duration_days coverage         : {round(ther['duration_days'].notna().mean() * 100, 2)}%")
print(ther["duration_source"].value_counts())

cache_path = os.path.join(CACHE_DIR, "clean_ther.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(ther, f)
print(f"\nCached cleaned THER -> {cache_path}")
print("Step 5 (THER cleaning) complete.")


Durations below the 1-day plausibility floor: 255,992 (31.58% of computed durations), set to NaN

THER CLEANING SUMMARY
Final row count                     : 1,956,680
Exact full-row duplicates removed    : 45700
Final duration_days coverage         : 28.34%
duration_source
unavailable                 1402165
from_both_reported_dates     442082
from_dur_cod_only            101606
from_derived_date             10827
Name: count, dtype: int64

Cached cleaned THER -> /content/drive/MyDrive/FAERS 2025/_cache/clean_ther.pkl
Step 5 (THER cleaning) complete.


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 6 — CLEANING: REAC, OUTC, RPSR, INDI
# ------------------------------------------------------------------------------
# Purpose : These four files are already narrow (Step 2's audit showed no
#           meaningful blank-field problems), so cleaning here is: remove only
#           EXACT full-row duplicates (a report can legitimately list the same
#           drug against several different reactions/outcomes/indications —
#           only fully-identical rows are redundant), validate coded fields
#           against the documented value sets, and flag INDI's non-informative
#           "unknown indication" placeholder for downstream exclusion in Q8.
# Input   : trimmed_reac.pkl, trimmed_outc.pkl, trimmed_rpsr.pkl,
#           trimmed_indi.pkl from Step 2.
# Output  : clean_reac.pkl, clean_outc.pkl, clean_rpsr.pkl, clean_indi.pkl
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load_trimmed(name):
    with open(os.path.join(CACHE_DIR, f"trimmed_{name}.pkl"), "rb") as f:
        return pickle.load(f)

reac = load_trimmed("reac")
outc = load_trimmed("outc")
rpsr = load_trimmed("rpsr")
indi = load_trimmed("indi")

for name, df in [("REAC", reac), ("OUTC", outc), ("RPSR", rpsr), ("INDI", indi)]:
    print(f"Loaded {name}: {df.shape}")

Loaded REAC: (5657830, 4)
Loaded OUTC: (1232582, 4)
Loaded RPSR: (43939, 4)
Loaded INDI: (4821589, 5)


In [ ]:
def remove_exact_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Removes only rows that are identical across every column (excluding
    source_quarter). Does NOT dedup on a subset key, since one primaryid can
    legitimately repeat with a different pt/outc_cod/indi_pt/rpsr_cod —
    that's multiple distinct reactions/outcomes/indications/sources for the
    same report, not redundancy.
    """
    compare_cols = [c for c in df.columns if c != "source_quarter"]
    dupe_mask = df.duplicated(subset=compare_cols, keep="first")
    n_dupes = dupe_mask.sum()
    print(f"{name}: {n_dupes:,} exact full-row duplicates removed")
    return df[~dupe_mask].reset_index(drop=True)

reac = remove_exact_duplicates(reac, "REAC")
outc = remove_exact_duplicates(outc, "OUTC")
rpsr = remove_exact_duplicates(rpsr, "RPSR")
indi = remove_exact_duplicates(indi, "INDI")

REAC: 70,717 exact full-row duplicates removed
OUTC: 124 exact full-row duplicates removed
RPSR: 57 exact full-row duplicates removed
INDI: 3,553 exact full-row duplicates removed


In [ ]:
# Keep the original pt (official MedDRA casing) for display, add an
# uppercased/stripped version for aggregation so Q7/Q9/Q13 groupings aren't
# fragmented by stray whitespace or casing variants.
reac["pt_clean"] = reac["pt"].astype(str).str.strip().str.upper()
reac["pt_clean"] = reac["pt_clean"].replace({"NAN": np.nan, "": np.nan})
print(f"\nREAC pt missing after cleaning: {reac['pt_clean'].isna().sum():,} "
      f"({round(reac['pt_clean'].isna().mean() * 100, 3)}%)")


REAC pt missing after cleaning: 0 (0.0%)


In [ ]:
VALID_OUTC = {"DE", "LT", "HO", "DS", "CA", "RI", "OT"}
outc_upper = outc["outc_cod"].astype(str).str.strip().str.upper()
unexpected_outc = ~outc_upper.isin(VALID_OUTC)
print(f"\nOUTC unexpected outc_cod values: {unexpected_outc.sum()} (flagged, not dropped)")
outc["outc_cod_clean"] = outc_upper


OUTC unexpected outc_cod values: 0 (flagged, not dropped)


In [ ]:
VALID_RPSR = {"FGN", "SDY", "LIT", "CSM", "HP", "UF", "CR", "DT", "OTH"}
rpsr_upper = rpsr["rpsr_cod"].astype(str).str.strip().str.upper()
unexpected_rpsr = ~rpsr_upper.isin(VALID_RPSR)
print(f"RPSR unexpected rpsr_cod values: {unexpected_rpsr.sum()} (flagged, not dropped)")
rpsr["rpsr_cod_clean"] = rpsr_upper

RPSR unexpected rpsr_cod values: 0 (flagged, not dropped)


In [ ]:
indi["indi_pt_clean"] = indi["indi_pt"].astype(str).str.strip().str.upper()
indi["indi_pt_clean"] = indi["indi_pt_clean"].replace({"NAN": np.nan, "": np.nan})

# Flagged separately (not dropped) since it's a legitimate, common MedDRA
# value — it's just non-informative for "what is this drug most commonly
# used for" in Q8, where it would otherwise dominate the ranking meaninglessly.
UNKNOWN_INDICATION_PLACEHOLDER = "PRODUCT USED FOR UNKNOWN INDICATION"
indi["is_unknown_indication"] = indi["indi_pt_clean"] == UNKNOWN_INDICATION_PLACEHOLDER

n_unknown = indi["is_unknown_indication"].sum()
print(f"\nINDI rows flagged as 'unknown indication' placeholder: {n_unknown:,} "
      f"({round(n_unknown / len(indi) * 100, 2)}% of INDI rows) — "
      f"kept in the data, flagged for exclusion specifically in the Q8 ranking")


INDI rows flagged as 'unknown indication' placeholder: 1,920,359 (39.86% of INDI rows) — kept in the data, flagged for exclusion specifically in the Q8 ranking


In [ ]:
print("\n" + "=" * 60)
print("STEP 6 CLEANING SUMMARY")
print("=" * 60)
for name, df in [("REAC", reac), ("OUTC", outc), ("RPSR", rpsr), ("INDI", indi)]:
    print(f"{name:6s} final row count: {len(df):,}")

for name, df in [("reac", reac), ("outc", outc), ("rpsr", rpsr), ("indi", indi)]:
    cache_path = os.path.join(CACHE_DIR, f"clean_{name}.pkl")
    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"Cached clean {name.upper()} -> {cache_path}")

print("\nStep 6 (REAC, OUTC, RPSR, INDI cleaning) complete.")
print("All 7 FAERS file types are now cleaned: DEMO, DRUG, THER, REAC, OUTC, RPSR, INDI.")


STEP 6 CLEANING SUMMARY
REAC   final row count: 5,587,113
OUTC   final row count: 1,232,458
RPSR   final row count: 43,882
INDI   final row count: 4,818,036
Cached clean REAC -> /content/drive/MyDrive/FAERS 2025/_cache/clean_reac.pkl
Cached clean OUTC -> /content/drive/MyDrive/FAERS 2025/_cache/clean_outc.pkl
Cached clean RPSR -> /content/drive/MyDrive/FAERS 2025/_cache/clean_rpsr.pkl
Cached clean INDI -> /content/drive/MyDrive/FAERS 2025/_cache/clean_indi.pkl

Step 6 (REAC, OUTC, RPSR, INDI cleaning) complete.
All 7 FAERS file types are now cleaned: DEMO, DRUG, THER, REAC, OUTC, RPSR, INDI.


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 7 — JOINING
# ------------------------------------------------------------------------------
# Purpose : Build the joined tables the 13 questions are actually computed
#           from. Deliberately NOT one flat merge of all 7 files — a single
#           report with N drugs x M reactions x K outcomes would fan out into
#           N*M*K duplicate-looking rows, corrupting every count-based
#           question. Instead: a safe report-level base (DEMO + outcome
#           flags), then several purpose-built tables joined at the grain
#           each question needs, using only many-to-one merges (except
#           drug_ther_level, where fan-out is a real, documented therapy
#           structure — see Step 5's Aricept note — not a join error).
# Input   : all 7 clean_*.pkl files from Steps 3-6.
# Output  : report_level.pkl, drug_level.pkl, reaction_level.pkl,
#           rpsr_level.pkl, drug_indi_level.pkl, drug_ther_level.pkl
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load_clean(name):
    with open(os.path.join(CACHE_DIR, f"clean_{name}.pkl"), "rb") as f:
        return pickle.load(f)

demo = load_clean("demo")
drug = load_clean("drug")
reac = load_clean("reac")
outc = load_clean("outc")
rpsr = load_clean("rpsr")
ther = load_clean("ther")
indi = load_clean("indi")

for name, df in [("DEMO", demo), ("DRUG", drug), ("REAC", reac), ("OUTC", outc),
                  ("RPSR", rpsr), ("THER", ther), ("INDI", indi)]:
    print(f"{name:6s}: {df.shape}")

DEMO  : (1617313, 25)
DRUG  : (7469819, 15)
REAC  : (5587113, 5)
OUTC  : (1232458, 5)
RPSR  : (43882, 5)
THER  : (1956680, 17)
INDI  : (4818036, 7)


In [ ]:
# A primaryid appears in OUTC ONLY if the event met a documented seriousness
# criterion (Section D.4: Death, Life-Threatening, Hospitalization, Disability,
# Congenital Anomaly, Required Intervention, or Other Serious). So presence in
# OUTC at all is itself the "serious outcome" flag — not something separate
# that needs inferring.
outcome_agg = outc.groupby("primaryid").agg(
    has_death=("outc_cod_clean", lambda s: (s == "DE").any()),
    n_distinct_outcomes=("outc_cod_clean", "nunique"),
).reset_index()
outcome_agg["has_serious_outcome"] = True   # by definition, every primaryid present in OUTC

n_serious = outcome_agg["has_serious_outcome"].sum()
print(f"\nReports with >=1 documented serious outcome: {n_serious:,} "
      f"({round(n_serious / len(demo) * 100, 2)}% of all reports)")


Reports with >=1 documented serious outcome: 919,627 (56.86% of all reports)


In [ ]:
report_level = demo.merge(outcome_agg, on="primaryid", how="left")
report_level["has_serious_outcome"] = report_level["has_serious_outcome"].fillna(False)
report_level["has_death"] = report_level["has_death"].fillna(False)
report_level["n_distinct_outcomes"] = report_level["n_distinct_outcomes"].fillna(0).astype(int)

assert len(report_level) == len(demo), "report_level row count drifted from DEMO — fan-out occurred, investigate"
print(f"\nreport_level: {report_level.shape} (should equal DEMO's {demo.shape[0]:,} rows exactly)")

# Columns actually needed downstream, kept lean so later merges don't drag
# along date-granularity/audit columns that were only needed during cleaning.
report_level_cols = [
    "primaryid", "caseid", "event_dt_clean", "mfr_dt_clean", "init_fda_dt_clean",
    "fda_dt_clean", "age_years", "age_grp", "sex", "reporter_country",
    "occr_country", "occp_cod", "has_serious_outcome", "has_death", "n_distinct_outcomes",
]
report_level = report_level[report_level_cols]

/tmp/ipykernel_8730/2281447586.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  report_level["has_serious_outcome"] = report_level["has_serious_outcome"].fillna(False)
/tmp/ipykernel_8730/2281447586.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  report_level["has_death"] = report_level["has_death"].fillna(False)



report_level: (1617313, 28) (should equal DEMO's 1,617,313 rows exactly)


In [ ]:
drug_cols = ["primaryid", "caseid", "drug_seq", "role_cod", "drug_label",
             "drug_label_source", "dechal_clean", "rechal_clean"]
drug_level = drug[drug_cols].merge(report_level, on="primaryid", how="left", suffixes=("", "_report"))

assert len(drug_level) == len(drug), "drug_level row count drifted from DRUG — unexpected fan-out, investigate"
print(f"drug_level: {drug_level.shape} (should equal cleaned DRUG's {drug.shape[0]:,} rows exactly)")

drug_level: (7469819, 22) (should equal cleaned DRUG's 7,469,819 rows exactly)


In [ ]:
reac_cols = ["primaryid", "caseid", "pt_clean"]
reaction_level = reac[reac_cols].merge(report_level, on="primaryid", how="left", suffixes=("", "_report"))

assert len(reaction_level) == len(reac), "reaction_level row count drifted from REAC — unexpected fan-out, investigate"
print(f"reaction_level: {reaction_level.shape} (should equal cleaned REAC's {reac.shape[0]:,} rows exactly)")

reaction_level: (5587113, 17) (should equal cleaned REAC's 5,587,113 rows exactly)


In [ ]:
rpsr_cols = ["primaryid", "caseid", "rpsr_cod_clean"]
rpsr_level = rpsr[rpsr_cols].merge(report_level, on="primaryid", how="left", suffixes=("", "_report"))

assert len(rpsr_level) == len(rpsr), "rpsr_level row count drifted from RPSR — unexpected fan-out, investigate"
print(f"rpsr_level: {rpsr_level.shape} (should equal cleaned RPSR's {rpsr.shape[0]:,} rows exactly)")

rpsr_level: (43882, 17) (should equal cleaned RPSR's 43,882 rows exactly)


In [ ]:
indi_cols = ["primaryid", "indi_drug_seq", "indi_pt_clean", "is_unknown_indication"]
drug_for_indi = drug[["primaryid", "drug_seq", "drug_label"]].rename(columns={"drug_seq": "join_seq"})
indi_for_join = indi[indi_cols].rename(columns={"indi_drug_seq": "join_seq"})

drug_indi_level = drug_for_indi.merge(
    indi_for_join, on=["primaryid", "join_seq"], how="inner"   # inner: only drug rows that have a matching indication entry
).rename(columns={"join_seq": "drug_seq"})

print(f"\ndrug_indi_level: {drug_indi_level.shape} "
      f"(inner join — rows exist only where a drug's drug_seq matches an INDI indi_drug_seq)")


drug_indi_level: (4818036, 5) (inner join — rows exist only where a drug's drug_seq matches an INDI indi_drug_seq)


In [ ]:
# Fan-out IS expected and legitimate here: a drug stopped and restarted (see
# Step 5's documented Aricept case) produces multiple genuine THER rows for
# the same drug_seq, and this table should reflect that.
ther_cols = ["primaryid", "dsg_drug_seq", "start_dt_clean", "end_dt_clean",
             "duration_days", "duration_source"]
drug_for_ther = drug[["primaryid", "drug_seq", "drug_label", "role_cod"]].rename(columns={"drug_seq": "join_seq"})
ther_for_join = ther[ther_cols].rename(columns={"dsg_drug_seq": "join_seq"})

drug_ther_level = drug_for_ther.merge(
    ther_for_join, on=["primaryid", "join_seq"], how="inner"
).rename(columns={"join_seq": "drug_seq"})

print(f"drug_ther_level: {drug_ther_level.shape} "
      f"(inner join — rows exist only where a drug has a matching therapy-date entry; "
      f"fan-out where a drug has multiple therapy periods is expected)")

drug_ther_level: (1956680, 8) (inner join — rows exist only where a drug has a matching therapy-date entry; fan-out where a drug has multiple therapy periods is expected)


In [ ]:
joined_tables = {
    "report_level": report_level,
    "drug_level": drug_level,
    "reaction_level": reaction_level,
    "rpsr_level": rpsr_level,
    "drug_indi_level": drug_indi_level,
    "drug_ther_level": drug_ther_level,
}

print("\n" + "=" * 60)
print("STEP 7 JOINING SUMMARY")
print("=" * 60)
for name, df in joined_tables.items():
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name:18s}: {df.shape[0]:>10,} rows | {df.shape[1]:2d} cols | {mem_mb:7.1f} MB")
    cache_path = os.path.join(CACHE_DIR, f"{name}.pkl")
    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"  cached -> {cache_path}")

print("\nStep 7 (Joining) complete. Note: Q7/Q13's drug-reaction pairwise table "
      "is intentionally NOT built here — it will be constructed and immediately "
      "aggregated inside the Phase 6 signal-detection step, not stored as a "
      "standing table, since it's the one genuinely combinatorial join in this project.")


STEP 7 JOINING SUMMARY
report_level      :  1,617,313 rows | 15 cols |   652.3 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/report_level.pkl
drug_level        :  7,469,819 rows | 22 cols |  5946.8 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/drug_level.pkl
reaction_level    :  5,587,113 rows | 17 cols |  2933.8 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/reaction_level.pkl
rpsr_level        :     43,882 rows | 17 cols |    21.4 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/rpsr_level.pkl
drug_indi_level   :  4,818,036 rows |  5 cols |  1194.7 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/drug_indi_level.pkl
drug_ther_level   :  1,956,680 rows |  8 cols |   606.9 MB
  cached -> /content/drive/MyDrive/FAERS 2025/_cache/drug_ther_level.pkl

Step 7 (Joining) complete. Note: Q7/Q13's drug-reaction pairwise table is intentionally NOT built here — it will be constructed and immediately aggregated inside the Phase 6 signal-detection step, 

In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 8 — ANSWERING QUESTIONS 1-6
# ------------------------------------------------------------------------------
# Purpose : Q1 (age/gender distribution), Q2 (event->FDA lag), Q3 (event->mfr
#           lag), Q4 (country concentration), Q5 (top drugs Pareto), Q6 (top
#           drugs' serious-outcome proportion) — all computable directly from
#           report_level and drug_level without needing the pairwise
#           drug-reaction join (Q7, Q13) or the therapy-duration join (Q12).
# Input   : report_level.pkl, drug_level.pkl from Step 7.
# Output  : question_results.pkl — a dict of result dataframes, one per
#           question, which Phase 7 will push to MySQL and Phase 9 will feed
#           to Tableau.
# NOTE: Restart the Colab runtime before running this step so old raw/trimmed
# dataframes from earlier steps aren't still occupying RAM alongside these
# large joined tables.
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

report_level = load("report_level")
drug_level = load("drug_level")

mem_before = (report_level.memory_usage(deep=True).sum() + drug_level.memory_usage(deep=True).sum()) / 1e6
print(f"Memory before category optimization: {mem_before:,.1f} MB")

# Low-cardinality repeated text columns -> category dtype. Doesn't change any
# values, just how pandas stores them; meaningfully reduces memory on tables
# this size.
CATEGORY_COLS_REPORT = ["sex", "age_grp", "reporter_country", "occr_country", "occp_cod"]
CATEGORY_COLS_DRUG = ["role_cod", "drug_label_source", "dechal_clean", "rechal_clean"]

for col in CATEGORY_COLS_REPORT:
    report_level[col] = report_level[col].astype("category")
for col in CATEGORY_COLS_DRUG:
    drug_level[col] = drug_level[col].astype("category")
drug_level["drug_label"] = drug_level["drug_label"].astype("category")

mem_after = (report_level.memory_usage(deep=True).sum() + drug_level.memory_usage(deep=True).sum()) / 1e6
print(f"Memory after category optimization:  {mem_after:,.1f} MB "
      f"({round((1 - mem_after / mem_before) * 100, 1)}% reduction)")

question_results = {}

Memory before category optimization: 6,599.1 MB
Memory after category optimization:  4,158.4 MB (37.0% reduction)


In [ ]:
# ==============================================================================
# Q1 — Age-group and gender distribution of ADR reports
# ==============================================================================
# Age bands match FDA's own age_grp definition (Section D: Neonate/Infant/
# Child/Adolescent/Adult/Elderly), derived here from age_years rather than
# read directly from age_grp, since age_years had far better coverage in
# Step 3's cleaning. age_grp is kept alongside as a cross-check, not the
# primary source.
AGE_BINS = [-0.001, 28 / 365.25, 2, 12, 18, 65, 130]
AGE_LABELS = ["Neonate", "Infant", "Child", "Adolescent", "Adult", "Elderly"]

report_level["age_group_derived"] = pd.cut(
    report_level["age_years"], bins=AGE_BINS, labels=AGE_LABELS
)
report_level["age_group_derived"] = report_level["age_group_derived"].cat.add_categories(["Not Reported"])
report_level["age_group_derived"] = report_level["age_group_derived"].fillna("Not Reported")

q1_age_sex = (
    report_level.groupby(["age_group_derived", "sex"], observed=True)
    .size()
    .reset_index(name="report_count")
)
q1_age_sex["pct_of_total"] = round(q1_age_sex["report_count"] / len(report_level) * 100, 2)
q1_age_sex = q1_age_sex.sort_values("report_count", ascending=False)

print("\n" + "=" * 60)
print("Q1 — Age-group x gender distribution")
print("=" * 60)
print(q1_age_sex.to_string(index=False))

# Cross-check: how often does our derived bucket agree with FDA's own age_grp?
FDA_CODE_TO_LABEL = {"N": "Neonate", "I": "Infant", "C": "Child", "T": "Adolescent", "A": "Adult", "E": "Elderly"}
comparable = report_level["age_grp"].notna() & (report_level["age_group_derived"] != "Not Reported")
agreement = (report_level.loc[comparable, "age_grp"].map(FDA_CODE_TO_LABEL)
             == report_level.loc[comparable, "age_group_derived"].astype(str))
print(f"\nCross-check vs FDA's own age_grp field: agreement on {round(agreement.mean() * 100, 2)}% "
      f"of the {comparable.sum():,} reports where both are available")

question_results["q1_age_gender_distribution"] = q1_age_sex


Q1 — Age-group x gender distribution
age_group_derived sex  report_count  pct_of_total
            Adult   F        318759         19.71
     Not Reported UNK        266874         16.50
     Not Reported   F        234902         14.52
          Elderly   F        187114         11.57
            Adult   M        186981         11.56
          Elderly   M        151822          9.39
     Not Reported   M        140133          8.66
            Adult UNK         29558          1.83
          Elderly UNK         19888          1.23
            Child   M         19658          1.22
       Adolescent   M         16006          0.99
            Child   F         15365          0.95
       Adolescent   F         15326          0.95
           Infant   M          4889          0.30
           Infant   F          3806          0.24
            Child UNK          1849          0.11
       Adolescent UNK          1535          0.09
           Infant UNK           947          0.06
          Ne

In [ ]:
# ==============================================================================
# Q2 — Reporting lag: event_dt -> fda_dt
# ==============================================================================
lag_fda = (report_level["fda_dt_clean"] - report_level["event_dt_clean"]).dt.days
both_present = report_level["fda_dt_clean"].notna() & report_level["event_dt_clean"].notna()
n_computable = both_present.sum()
print(f"\n" + "=" * 60)
print("Q2 — Reporting lag (event -> FDA)")
print("=" * 60)
print(f"Computable for {n_computable:,} of {len(report_level):,} reports "
      f"({round(n_computable / len(report_level) * 100, 2)}%) — the rest are missing event_dt (56.28% per Step 3)")

negative_lag = both_present & (lag_fda < 0)
print(f"Negative lag (FDA received report before the event date — implausible): "
      f"{negative_lag.sum():,}, excluded from the statistics below")
lag_fda_valid = lag_fda[both_present & ~negative_lag]

print(f"Mean lag   : {lag_fda_valid.mean():.1f} days")
print(f"Median lag : {lag_fda_valid.median():.1f} days")
print(f"90th pctile: {lag_fda_valid.quantile(0.90):.1f} days")

question_results["q2_reporting_lag_to_fda"] = pd.DataFrame({
    "metric": ["mean_days", "median_days", "p90_days", "n_computable", "n_negative_excluded"],
    "value": [lag_fda_valid.mean(), lag_fda_valid.median(), lag_fda_valid.quantile(0.90),
              n_computable, negative_lag.sum()],
})


Q2 — Reporting lag (event -> FDA)
Computable for 707,054 of 1,617,313 reports (43.72%) — the rest are missing event_dt (56.28% per Step 3)
Negative lag (FDA received report before the event date — implausible): 12, excluded from the statistics below
Mean lag   : 403.4 days
Median lag : 138.0 days
90th pctile: 1019.0 days


In [ ]:
# ==============================================================================
# Q3 — Average time between event and manufacturer notification: event_dt -> mfr_dt
# ==============================================================================
lag_mfr = (report_level["mfr_dt_clean"] - report_level["event_dt_clean"]).dt.days
both_present_mfr = report_level["mfr_dt_clean"].notna() & report_level["event_dt_clean"].notna()
n_computable_mfr = both_present_mfr.sum()
print(f"\n" + "=" * 60)
print("Q3 — Reporting lag (event -> manufacturer notification)")
print("=" * 60)
print(f"Computable for {n_computable_mfr:,} of {len(report_level):,} reports "
      f"({round(n_computable_mfr / len(report_level) * 100, 2)}%)")

negative_lag_mfr = both_present_mfr & (lag_mfr < 0)
print(f"Negative lag (implausible): {negative_lag_mfr.sum():,}, excluded")
lag_mfr_valid = lag_mfr[both_present_mfr & ~negative_lag_mfr]

print(f"Mean lag  : {lag_mfr_valid.mean():.1f} days")
print(f"Median lag: {lag_mfr_valid.median():.1f} days")

question_results["q3_reporting_lag_to_manufacturer"] = pd.DataFrame({
    "metric": ["mean_days", "median_days", "n_computable", "n_negative_excluded"],
    "value": [lag_mfr_valid.mean(), lag_mfr_valid.median(), n_computable_mfr, negative_lag_mfr.sum()],
})


Q3 — Reporting lag (event -> manufacturer notification)
Computable for 707,052 of 1,617,313 reports (43.72%)
Negative lag (implausible): 755, excluded
Mean lag  : 370.8 days
Median lag: 99.0 days


In [ ]:
# ==============================================================================
# Q4 — Country volume concentration (top 5 as % of total)
# ==============================================================================
# reporter_country = who submitted the report; occr_country = where the event
# happened. "Which countries GENERATE reports" reads most naturally as
# reporter_country — used as the primary answer, with occr_country shown
# alongside since it answers a related but distinct question.
print(f"\n" + "=" * 60)
print("Q4 — Country concentration (top 5 as % of total)")
print("=" * 60)

for country_col, label in [("reporter_country", "Reporter country"), ("occr_country", "Occurrence country")]:
    counts = report_level[country_col].astype(str).str.strip().str.upper()
    counts = counts.replace({"NAN": "UNKNOWN/NOT REPORTED", "": "UNKNOWN/NOT REPORTED"})
    vc = counts.value_counts()
    known_total = vc.drop("UNKNOWN/NOT REPORTED", errors="ignore").sum()
    top5 = vc.drop("UNKNOWN/NOT REPORTED", errors="ignore").head(5)
    top5_pct_of_known = round(top5.sum() / known_total * 100, 2)
    print(f"\n{label} — top 5 (% of reports with a known country, n={known_total:,}):")
    print((top5 / known_total * 100).round(2))
    print(f"Top 5 combined: {top5_pct_of_known}% of known-country reports")
    print(f"Unknown/not reported: {vc.get('UNKNOWN/NOT REPORTED', 0):,} "
          f"({round(vc.get('UNKNOWN/NOT REPORTED', 0) / len(report_level) * 100, 2)}% of all reports)")

    if country_col == "reporter_country":
        question_results["q4_country_concentration"] = pd.DataFrame({
            "country": top5.index, "report_count": top5.values,
            "pct_of_known_total": (top5 / known_total * 100).round(2).values,
        })


Q4 — Country concentration (top 5 as % of total)

Reporter country — top 5 (% of reports with a known country, n=1,617,311):
reporter_country
US    65.65
CA     6.52
EU     5.59
JP     4.07
GB     3.74
Name: count, dtype: float64
Top 5 combined: 85.56% of known-country reports
Unknown/not reported: 2 (0.0% of all reports)

Occurrence country — top 5 (% of reports with a known country, n=1,537,750):
occr_country
US    64.31
CA     6.75
EU     5.81
JP     4.28
GB     3.89
Name: count, dtype: float64
Top 5 combined: 85.04% of known-country reports
Unknown/not reported: 79,563 (4.92% of all reports)


In [ ]:
# ==============================================================================
# Q5 — Top drugs by prod_ai (Pareto: top 10 as % of total)
# ==============================================================================
# Suspect-only (PS+SS) is the standard convention for "which drugs generate
# reports" — a concomitant ('C') drug is just something the patient happened
# to also be taking, not implicated in the event, so including it would
# inflate the ranking with drugs that aren't actually the subject of the ADR.
# Both views are computed for transparency; suspect-only is the primary answer.
print(f"\n" + "=" * 60)
print("Q5 — Top drugs by prod_ai (Pareto)")
print("=" * 60)

for scope_name, scope_mask in [
    ("All roles", pd.Series(True, index=drug_level.index)),
    ("Suspect only (PS+SS)", drug_level["role_cod"].isin(["PS", "SS"])),
]:
    scoped = drug_level[scope_mask]
    vc = scoped["drug_label"].value_counts()
    total = vc.sum()
    top10 = vc.head(10)
    top10_pct = round(top10.sum() / total * 100, 2)
    print(f"\n{scope_name} (n={total:,} drug entries):")
    print((top10 / total * 100).round(2))
    print(f"Top 10 combined: {top10_pct}% of {scope_name.lower()} drug volume")

    if scope_name.startswith("Suspect"):
        top10_drugs = top10.index.tolist()
        question_results["q5_top_drugs_pareto"] = pd.DataFrame({
            "drug_label": top10.index, "report_count": top10.values,
            "pct_of_suspect_total": (top10 / total * 100).round(2).values,
        })


Q5 — Top drugs by prod_ai (Pareto)

All roles (n=7,469,819 drug entries):
drug_label
TIRZEPATIDE        3.16
DUPILUMAB          2.67
PREDNISONE         1.37
INFLIXIMAB-DYYB    1.29
ACETAMINOPHEN      1.20
METHOTREXATE       1.18
RITUXIMAB          1.04
TOCILIZUMAB        0.95
INFLIXIMAB         0.92
ADALIMUMAB         0.76
Name: count, dtype: float64
Top 10 combined: 14.54% of all roles drug volume

Suspect only (PS+SS) (n=4,603,624 drug entries):
drug_label
TIRZEPATIDE        5.04
DUPILUMAB          4.30
INFLIXIMAB-DYYB    2.08
RITUXIMAB          1.52
METHOTREXATE       1.50
INFLIXIMAB         1.39
TOCILIZUMAB        1.38
PREDNISONE         1.29
ADALIMUMAB         1.12
OMALIZUMAB         0.85
Name: count, dtype: float64
Top 10 combined: 20.47% of suspect only (ps+ss) drug volume


In [ ]:
# ==============================================================================
# Q6 — For the top-reported drugs, proportion of reports linked to serious outcomes
# ==============================================================================
print(f"\n" + "=" * 60)
print("Q6 — Serious-outcome proportion for Q5's top 10 drugs")
print("=" * 60)

suspect_drugs = drug_level[drug_level["role_cod"].isin(["PS", "SS"])]
top10_subset = suspect_drugs[suspect_drugs["drug_label"].isin(top10_drugs)]

q6 = (
    top10_subset.groupby("drug_label", observed=True)
    .agg(total_reports=("primaryid", "nunique"),
         serious_reports=("has_serious_outcome", "sum"))
    .reset_index()
)
q6["serious_pct"] = round(q6["serious_reports"] / q6["total_reports"] * 100, 2)
q6 = q6.sort_values("total_reports", ascending=False)
print(q6.to_string(index=False))

question_results["q6_top_drugs_serious_outcome_pct"] = q6


Q6 — Serious-outcome proportion for Q5's top 10 drugs
     drug_label  total_reports  serious_reports  serious_pct
      DUPILUMAB         143867            21120        14.68
    TIRZEPATIDE          60763            29719        48.91
     ADALIMUMAB          24112            45962       190.62
     PREDNISONE          23948            57709       240.98
      RITUXIMAB          23574            68647       291.20
   METHOTREXATE          17615            68367       388.12
     INFLIXIMAB          14920            63372       424.75
     OMALIZUMAB          10841            29940       276.17
    TOCILIZUMAB          10043            60902       606.41
INFLIXIMAB-DYYB           8368            94670      1131.33


In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)

print("\n" + "=" * 60)
print(f"Step 8 complete. Cached {len(question_results)} result tables -> {cache_path}")
print("=" * 60)
for k in question_results:
    print(f"  - {k}")


Step 8 complete. Cached 6 result tables -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl
  - q1_age_gender_distribution
  - q2_reporting_lag_to_fda
  - q3_reporting_lag_to_manufacturer
  - q4_country_concentration
  - q5_top_drugs_pareto
  - q6_top_drugs_serious_outcome_pct


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 9 — ANSWERING QUESTIONS 8-13
# ------------------------------------------------------------------------------
# Purpose : Q8 (indication for top drugs), Q9 (reactions most linked to serious
#           outcomes), Q10 (country death RATE, not raw count), Q11 (reporter
#           source vs severity), Q12 (therapy duration vs severity for a
#           selected high-volume drug), Q13 (dechal/rechal for top-reported
#           drug-reaction pairs).
# Input   : question_results.pkl, report_level.pkl, drug_level.pkl,
#           reaction_level.pkl, rpsr_level.pkl, drug_indi_level.pkl,
#           drug_ther_level.pkl
# Output  : question_results.pkl, updated with q8-q13 result tables.
# NOTE: Q7's PRR/ROR signal detection is intentionally NOT included here —
# it belongs to its own dedicated Phase 6 step, separate from these six.
# ==============================================================================
import os
import gc
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
report_level = load("report_level")
drug_level = load("drug_level")
reaction_level = load("reaction_level")
rpsr_level = load("rpsr_level")
drug_indi_level = load("drug_indi_level")
drug_ther_level = load("drug_ther_level")

top10_drugs = question_results["q5_top_drugs_pareto"]["drug_label"].tolist()
print(f"Top 10 suspect drugs from Q5: {top10_drugs}")

Top 10 suspect drugs from Q5: ['TIRZEPATIDE', 'DUPILUMAB', 'INFLIXIMAB-DYYB', 'RITUXIMAB', 'METHOTREXATE', 'INFLIXIMAB', 'TOCILIZUMAB', 'PREDNISONE', 'ADALIMUMAB', 'OMALIZUMAB']


In [ ]:
# ==============================================================================
# Q8 — Most common indication for the top drugs
# ==============================================================================
# Excludes the "Product used for unknown indication" placeholder (flagged in
# Step 6) from the ranking itself, since it's not a real medical indication —
# but reports what share of each drug's indication entries it makes up, so
# that context isn't lost.
print("\n" + "=" * 60)
print("Q8 — Most common indication for top drugs")
print("=" * 60)

q8_rows = []
for d in top10_drugs:
    sub = drug_indi_level[drug_indi_level["drug_label"] == d]
    total_entries = len(sub)
    placeholder_pct = round(sub["is_unknown_indication"].mean() * 100, 2) if total_entries else np.nan
    real = sub[~sub["is_unknown_indication"]]
    if len(real):
        vc = real["indi_pt_clean"].value_counts()
        top_indi, top_indi_count = vc.idxmax(), vc.max()
    else:
        top_indi, top_indi_count = "N/A", 0
    q8_rows.append({
        "drug_label": d, "total_indication_entries": total_entries,
        "unknown_indication_pct": placeholder_pct,
        "top_real_indication": top_indi, "top_indication_report_count": top_indi_count,
    })

q8 = pd.DataFrame(q8_rows)
print(q8.to_string(index=False))
question_results["q8_top_indication_per_drug"] = q8


Q8 — Most common indication for top drugs
     drug_label  total_indication_entries  unknown_indication_pct  top_real_indication  top_indication_report_count
    TIRZEPATIDE                    199989                   49.32       WEIGHT CONTROL                        50643
      DUPILUMAB                    157252                    1.19    DERMATITIS ATOPIC                        64855
INFLIXIMAB-DYYB                      9111                    5.13      CROHN'S DISEASE                         4793
      RITUXIMAB                     36539                   10.85 RHEUMATOID ARTHRITIS                        10658
   METHOTREXATE                     34625                   25.42 RHEUMATOID ARTHRITIS                        12991
     INFLIXIMAB                     26370                   14.61 RHEUMATOID ARTHRITIS                         7667
    TOCILIZUMAB                     21912                   16.99 RHEUMATOID ARTHRITIS                        14008
     PREDNISONE              

In [ ]:
# ==============================================================================
# Q9 — Reactions most frequently linked to serious outcomes (top 10, Pareto)
# ==============================================================================
print("\n" + "=" * 60)
print("Q9 — Top 10 reactions linked to serious outcomes")
print("=" * 60)

serious_reac = reaction_level[reaction_level["has_serious_outcome"]]
vc = serious_reac["pt_clean"].value_counts()
total = vc.sum()
top10_reac = vc.head(10)

q9 = pd.DataFrame({
    "reaction": top10_reac.index, "count": top10_reac.values,
    "pct_of_serious_reaction_listings": (top10_reac / total * 100).round(2).values,
})
print(f"Total reaction listings on serious-outcome reports: {total:,}")
print(q9.to_string(index=False))
print(f"Top 10 combined: {round(top10_reac.sum() / total * 100, 2)}% of all serious-outcome reaction listings")
question_results["q9_top_reactions_serious_outcome"] = q9


Q9 — Top 10 reactions linked to serious outcomes
Total reaction listings on serious-outcome reports: 3,928,089
        reaction  count  pct_of_serious_reaction_listings
   OFF LABEL USE  74167                              1.89
           DEATH  59696                              1.52
DRUG INEFFECTIVE  46753                              1.19
         FATIGUE  43675                              1.11
       DIARRHOEA  39328                              1.00
          NAUSEA  38773                              0.99
       PNEUMONIA  32456                              0.83
        DYSPNOEA  31763                              0.81
        VOMITING  30073                              0.77
            PAIN  29866                              0.76
Top 10 combined: 10.86% of all serious-outcome reaction listings


In [ ]:
# ==============================================================================
# Q10 — Countries with highest DEATH RATE (not raw count)
# ==============================================================================
# Minimum-report threshold applied so a country with e.g. 2 reports and 1
# death doesn't rank #1 at a "50% death rate" — this is the same
# small-sample-distortion logic as the PRR minimum case count.
MIN_REPORTS_Q10 = 100
print("\n" + "=" * 60)
print(f"Q10 — Country death rate (min {MIN_REPORTS_Q10} reports to qualify)")
print("=" * 60)

country = report_level.copy()
country["reporter_country"] = country["reporter_country"].astype(str).str.strip().str.upper()
country = country[~country["reporter_country"].isin(["", "NAN"])]

q10_full = country.groupby("reporter_country").agg(
    total_reports=("primaryid", "size"),
    death_reports=("has_death", "sum"),
).reset_index()
q10_full["death_rate_pct"] = round(q10_full["death_reports"] / q10_full["total_reports"] * 100, 2)

n_excluded_countries = (q10_full["total_reports"] < MIN_REPORTS_Q10).sum()
print(f"{n_excluded_countries} countries excluded for having fewer than {MIN_REPORTS_Q10} reports")

q10 = q10_full[q10_full["total_reports"] >= MIN_REPORTS_Q10].sort_values(
    "death_rate_pct", ascending=False
).head(10).reset_index(drop=True)
print(q10.to_string(index=False))
question_results["q10_country_death_rate"] = q10


Q10 — Country death rate (min 100 reports to qualify)
112 countries excluded for having fewer than 100 reports
reporter_country  total_reports  death_reports  death_rate_pct
              KE            158             98           62.03
              MA            710            399           56.20
              NP            274            149           54.38
              ID            839            399           47.56
              PK           1491            677           45.41
              PH            860            382           44.42
              BY            114             40           35.09
              HK            409            142           34.72
              VN           1042            354           33.97
              DZ            175             58           33.14


In [ ]:
# ==============================================================================
# Q11 — Primary reporters vs. severity of outcome
# ==============================================================================
# Reminder from Step 2's audit: RPSR only covers ~2.7% of all reports, so
# this answers "among the reports that DO record a source, how does severity
# vary by source" — not a population-wide claim.
print("\n" + "=" * 60)
print("Q11 — Reporter source vs. severity (RPSR coverage: ~2.7% of all reports)")
print("=" * 60)

q11 = rpsr_level.groupby("rpsr_cod_clean").agg(
    total_reports=("primaryid", "size"),
    serious_reports=("has_serious_outcome", "sum"),
    death_reports=("has_death", "sum"),
).reset_index()
q11["serious_pct"] = round(q11["serious_reports"] / q11["total_reports"] * 100, 2)
q11["death_pct"] = round(q11["death_reports"] / q11["total_reports"] * 100, 2)
q11 = q11.sort_values("total_reports", ascending=False)
print(q11.to_string(index=False))
question_results["q11_reporter_source_vs_severity"] = q11


Q11 — Reporter source vs. severity (RPSR coverage: ~2.7% of all reports)
rpsr_cod_clean  total_reports  serious_reports  death_reports  serious_pct  death_pct
           CSM          24282            12958           1284        53.36       5.29
            HP          19132            13048           2809        68.20      14.68
           FGN            468              407             10        86.97       2.14


In [ ]:
# ==============================================================================
# Q12 — Therapy duration vs. severity, for a selected high-volume drug
# ==============================================================================
# Defaults to Q5's #1 ranked suspect drug — change SELECTED_DRUG to analyze
# a different one. drug_ther_level didn't carry outcome flags from Step 7
# (it wasn't needed by any other question), so they're merged in here.
SELECTED_DRUG = top10_drugs[0]
print("\n" + "=" * 60)
print(f"Q12 — Therapy duration vs. severity for: {SELECTED_DRUG}")
print("=" * 60)

ther_scope = drug_ther_level[
    (drug_ther_level["drug_label"] == SELECTED_DRUG) &
    (drug_ther_level["role_cod"].isin(["PS", "SS"]))
].merge(report_level[["primaryid", "has_serious_outcome"]], on="primaryid", how="left")

ther_scope = ther_scope[ther_scope["duration_days"].notna()]
print(f"Rows with a computable duration for {SELECTED_DRUG}: {len(ther_scope):,}")

DURATION_BINS = [0, 7, 30, 90, 180, 365, 1_000_000]
DURATION_LABELS = ["<=7d", "8-30d", "31-90d", "91-180d", "181-365d", ">365d"]
ther_scope["duration_bucket"] = pd.cut(
    ther_scope["duration_days"], bins=DURATION_BINS, labels=DURATION_LABELS, include_lowest=True
)

q12 = ther_scope.groupby("duration_bucket", observed=True).agg(
    n_reports=("primaryid", "size"),
    serious_pct=("has_serious_outcome", lambda s: round(s.mean() * 100, 2)),
).reset_index()
print(q12.to_string(index=False))
question_results["q12_duration_vs_severity"] = q12
question_results["q12_selected_drug"] = SELECTED_DRUG


Q12 — Therapy duration vs. severity for: TIRZEPATIDE
Rows with a computable duration for TIRZEPATIDE: 7,586
duration_bucket  n_reports  serious_pct
           <=7d        719        64.39
          8-30d       1981        67.54
         31-90d       2337        76.77
        91-180d       1244        76.29
       181-365d        878        76.08
          >365d        427        75.64


In [ ]:
# ==============================================================================
# STEP 9 (Q13 FIX) — memory-safe drug-reaction pairwise aggregation
# ------------------------------------------------------------------------------
# Replaces the original Q13 cells. The crash was almost certainly caused by
# outlier reports with unusually many listed drugs AND reactions — a report
# with 50 suspect drugs x 50 reactions contributes 2,500 rows to the merge on
# its own. A few such reports among 1.6M can exhaust RAM even though the
# average case is small. Fix: (1) exclude extreme outlier reports from the
# pairwise analysis, with the exclusion counted and documented, and (2)
# process the merge in batches of reports so peak memory never depends on the
# total combinatorial size, only on one batch at a time.
# ==============================================================================
import os
import gc
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
drug_level = load("drug_level")
reaction_level = load("reaction_level")

In [ ]:
suspect_small = drug_level[drug_level["role_cod"].isin(["PS", "SS"])][
    ["primaryid", "drug_label", "dechal_clean", "rechal_clean"]
].copy()
reac_small = reaction_level[["primaryid", "pt_clean"]].copy()

print(f"suspect_small: {suspect_small.shape}, reac_small: {reac_small.shape}")

suspect_small: (4603624, 4), reac_small: (5587113, 2)


In [ ]:
# A report's contribution to the pairwise merge is (its suspect-drug count) x
# (its reaction count). Cap that product — reports above the cap are excluded
# from THIS analysis only (not from any other question), with the count
# documented so it goes in the report's limitations section.
drug_counts = suspect_small.groupby("primaryid").size()
reac_counts = reac_small.groupby("primaryid").size()

combo_size = (drug_counts.reindex(drug_counts.index.union(reac_counts.index), fill_value=0) *
              reac_counts.reindex(drug_counts.index.union(reac_counts.index), fill_value=0))

FANOUT_CAP = 400   # e.g. 20 drugs x 20 reactions — generous for a normal report, tight enough to block outliers
outlier_pids = combo_size[combo_size > FANOUT_CAP].index
print(f"Reports excluded from Q13 for exceeding a {FANOUT_CAP}-row fan-out cap: {len(outlier_pids):,} "
      f"({round(len(outlier_pids) / len(combo_size) * 100, 3)}% of reports with suspect drugs and/or reactions)")

suspect_small = suspect_small[~suspect_small["primaryid"].isin(outlier_pids)]
reac_small = reac_small[~reac_small["primaryid"].isin(outlier_pids)]

del drug_counts, reac_counts, combo_size
gc.collect()

Reports excluded from Q13 for exceeding a 400-row fan-out cap: 12,074 (0.747% of reports with suspect drugs and/or reactions)


14

In [ ]:
MIN_PAIR_COUNT_Q13 = 20
BATCH_SIZE = 50_000   # unique primaryids per batch — lower this (e.g. 10_000) if it still crashes

unique_pids = suspect_small["primaryid"].unique()
n_batches = (len(unique_pids) // BATCH_SIZE) + 1
print(f"Processing {len(unique_pids):,} primaryids in {n_batches} batches of ~{BATCH_SIZE:,}")

accumulator = {}   # (drug_label, pt_clean) -> [pair_count, dechal_pos, rechal_pos]

for i in range(0, len(unique_pids), BATCH_SIZE):
    batch_ids = set(unique_pids[i:i + BATCH_SIZE])

    drug_chunk = suspect_small[suspect_small["primaryid"].isin(batch_ids)]
    reac_chunk = reac_small[reac_small["primaryid"].isin(batch_ids)]
    merged_chunk = drug_chunk.merge(reac_chunk, on="primaryid", how="inner")

    if len(merged_chunk):
        grp = merged_chunk.groupby(["drug_label", "pt_clean"], observed=True).agg(
            pair_count=("primaryid", "size"),
            dechal_pos=("dechal_clean", lambda s: (s == "Y").sum()),
            rechal_pos=("rechal_clean", lambda s: (s == "Y").sum()),
        ).reset_index()

        for row in grp.itertuples(index=False):
            key = (row.drug_label, row.pt_clean)
            if key not in accumulator:
                accumulator[key] = [0, 0, 0]
            accumulator[key][0] += row.pair_count
            accumulator[key][1] += row.dechal_pos
            accumulator[key][2] += row.rechal_pos

    del drug_chunk, reac_chunk, merged_chunk
    gc.collect()
    print(f"  batch {i // BATCH_SIZE + 1}/{n_batches} done, {len(accumulator):,} distinct pairs so far")

print(f"\nBatched merge complete. Total distinct drug-reaction pairs seen: {len(accumulator):,}")

Processing 1,605,229 primaryids in 33 batches of ~50,000
  batch 1/33 done, 94,134 distinct pairs so far
  batch 2/33 done, 162,408 distinct pairs so far
  batch 3/33 done, 214,257 distinct pairs so far
  batch 4/33 done, 261,910 distinct pairs so far
  batch 5/33 done, 301,902 distinct pairs so far
  batch 6/33 done, 406,057 distinct pairs so far
  batch 7/33 done, 442,006 distinct pairs so far
  batch 8/33 done, 488,002 distinct pairs so far
  batch 9/33 done, 518,338 distinct pairs so far
  batch 10/33 done, 549,394 distinct pairs so far
  batch 11/33 done, 578,686 distinct pairs so far
  batch 12/33 done, 606,577 distinct pairs so far
  batch 13/33 done, 630,073 distinct pairs so far
  batch 14/33 done, 690,187 distinct pairs so far
  batch 15/33 done, 712,140 distinct pairs so far
  batch 16/33 done, 733,398 distinct pairs so far
  batch 17/33 done, 759,595 distinct pairs so far
  batch 18/33 done, 779,010 distinct pairs so far
  batch 19/33 done, 800,896 distinct pairs so far
  b

In [ ]:
rows = []
for (drug, pt), (count, dechal_pos, rechal_pos) in accumulator.items():
    rows.append({
        "drug_label": drug, "pt_clean": pt, "pair_count": count,
        "pct_dechal_positive": round(dechal_pos / count * 100, 2),
        "pct_rechal_positive": round(rechal_pos / count * 100, 2),
    })

pair_stats = pd.DataFrame(rows)
pair_stats = pair_stats[pair_stats["pair_count"] >= MIN_PAIR_COUNT_Q13]
q13 = pair_stats.sort_values("pair_count", ascending=False).head(10).reset_index(drop=True)

print("\n" + "=" * 60)
print(f"Q13 — Dechal/rechal for top drug-reaction pairs (min {MIN_PAIR_COUNT_Q13} co-occurrences)")
print("=" * 60)
print(q13.to_string(index=False))

question_results["q13_dechal_rechal_top_pairs"] = q13
question_results["q13_excluded_outlier_reports"] = len(outlier_pids)


Q13 — Dechal/rechal for top drug-reaction pairs (min 20 co-occurrences)
     drug_label                             pt_clean  pair_count  pct_dechal_positive  pct_rechal_positive
    TIRZEPATIDE          INCORRECT DOSE ADMINISTERED       59857                 0.48                 0.01
INFLIXIMAB-DYYB                 CONDITION AGGRAVATED       38049                 7.83                 0.00
INFLIXIMAB-DYYB                        OFF LABEL USE       37684                 8.10                 0.00
      DUPILUMAB                             PRURITUS       32033                 5.64                 0.61
    TIRZEPATIDE                  INJECTION SITE PAIN       28253                 1.07                 0.07
    TIRZEPATIDE                               NAUSEA       26724                13.89                 0.17
      DUPILUMAB                    DERMATITIS ATOPIC       25210                 3.17                 0.34
      DUPILUMAB PRODUCT USE IN UNAPPROVED INDICATION       22178       

In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)
print(f"\nCached updated question_results -> {cache_path}")


Cached updated question_results -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# Q7 — DISPROPORTIONALITY SIGNAL DETECTION (PRR / ROR)
# ------------------------------------------------------------------------------
# Purpose : For every suspect drug-reaction pair with sufficient volume,
#           build a 2x2 contingency table and compute PRR, ROR, and a
#           chi-square statistic, then flag genuine signals using the
#           standard Evans et al. (2001) criteria: PRR >= 2, chi-square >= 4,
#           and at least 3 cases. Ranks the flagged signals by PRR.
# Input   : drug_level.pkl, reaction_level.pkl
# Output  : question_results.pkl, updated with q7_prr_ror_signals
# NOTE: reuses the same outlier-report cap + batched merge from the Q13 fix,
# since Q7 needs the identical drug x reaction cross-join and has the same
# combinatorial crash risk.
# ==============================================================================
import os
import gc
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
drug_level = load("drug_level")
reaction_level = load("reaction_level")

suspect_small = drug_level[drug_level["role_cod"].isin(["PS", "SS"])][["primaryid", "drug_label"]].copy()
reac_small = reaction_level[["primaryid", "pt_clean"]].copy()
print(f"suspect_small: {suspect_small.shape}, reac_small: {reac_small.shape}")

suspect_small: (4603624, 2), reac_small: (5587113, 2)


In [ ]:
drug_counts = suspect_small.groupby("primaryid").size()
reac_counts = reac_small.groupby("primaryid").size()
all_pids = drug_counts.index.union(reac_counts.index)
combo_size = (drug_counts.reindex(all_pids, fill_value=0) * reac_counts.reindex(all_pids, fill_value=0))

FANOUT_CAP = 400
outlier_pids = combo_size[combo_size > FANOUT_CAP].index
print(f"Reports excluded for exceeding a {FANOUT_CAP}-row fan-out cap: {len(outlier_pids):,}")

suspect_small = suspect_small[~suspect_small["primaryid"].isin(outlier_pids)]
reac_small = reac_small[~reac_small["primaryid"].isin(outlier_pids)]
del drug_counts, reac_counts, all_pids, combo_size
gc.collect()

Reports excluded for exceeding a 400-row fan-out cap: 12,074


7

In [ ]:
# n_drug[D]     = reports containing drug D (within this analysis population)
# n_reaction[R] = reports containing reaction R (within this analysis population)
# N             = total reports in the analysis population (union of both files' primaryids)
n_drug = suspect_small.groupby("drug_label", observed=True)["primaryid"].nunique()
n_reaction = reac_small.groupby("pt_clean", observed=True)["primaryid"].nunique()
N = len(set(suspect_small["primaryid"]).union(set(reac_small["primaryid"])))
print(f"Analysis population N = {N:,} reports | {len(n_drug):,} distinct suspect drugs | {len(n_reaction):,} distinct reactions")

Analysis population N = 1,605,239 reports | 6,487 distinct suspect drugs | 16,720 distinct reactions


In [ ]:
BATCH_SIZE = 50_000
unique_pids = suspect_small["primaryid"].unique()
n_batches = (len(unique_pids) // BATCH_SIZE) + 1
print(f"Processing {len(unique_pids):,} primaryids in {n_batches} batches of ~{BATCH_SIZE:,}")

accumulator = {}   # (drug_label, pt_clean) -> co-occurrence count (a)

for i in range(0, len(unique_pids), BATCH_SIZE):
    batch_ids = set(unique_pids[i:i + BATCH_SIZE])
    drug_chunk = suspect_small[suspect_small["primaryid"].isin(batch_ids)]
    reac_chunk = reac_small[reac_small["primaryid"].isin(batch_ids)]
    merged_chunk = drug_chunk.merge(reac_chunk, on="primaryid", how="inner")

    if len(merged_chunk):
        counts = merged_chunk.groupby(["drug_label", "pt_clean"], observed=True).size()
        for key, val in counts.items():
            accumulator[key] = accumulator.get(key, 0) + val

    del drug_chunk, reac_chunk, merged_chunk
    gc.collect()
    print(f"  batch {i // BATCH_SIZE + 1}/{n_batches} done, {len(accumulator):,} distinct pairs so far")

print(f"\nTotal distinct drug-reaction pairs: {len(accumulator):,}")

Processing 1,605,229 primaryids in 33 batches of ~50,000
  batch 1/33 done, 94,134 distinct pairs so far
  batch 2/33 done, 162,408 distinct pairs so far
  batch 3/33 done, 214,257 distinct pairs so far
  batch 4/33 done, 261,910 distinct pairs so far
  batch 5/33 done, 301,902 distinct pairs so far
  batch 6/33 done, 406,057 distinct pairs so far
  batch 7/33 done, 442,006 distinct pairs so far
  batch 8/33 done, 488,002 distinct pairs so far
  batch 9/33 done, 518,338 distinct pairs so far
  batch 10/33 done, 549,394 distinct pairs so far
  batch 11/33 done, 578,686 distinct pairs so far
  batch 12/33 done, 606,577 distinct pairs so far
  batch 13/33 done, 630,073 distinct pairs so far
  batch 14/33 done, 690,187 distinct pairs so far
  batch 15/33 done, 712,140 distinct pairs so far
  batch 16/33 done, 733,398 distinct pairs so far
  batch 17/33 done, 759,595 distinct pairs so far
  batch 18/33 done, 779,010 distinct pairs so far
  batch 19/33 done, 800,896 distinct pairs so far
  b

In [ ]:
pairs = pd.DataFrame(
    [(d, r, a) for (d, r), a in accumulator.items()],
    columns=["drug_label", "pt_clean", "a"]
)
pairs["n_drug"] = pairs["drug_label"].map(n_drug)
pairs["n_reaction"] = pairs["pt_clean"].map(n_reaction)

pairs["b"] = pairs["n_drug"] - pairs["a"]
pairs["c"] = pairs["n_reaction"] - pairs["a"]
pairs["d"] = N - pairs["n_drug"] - pairs["n_reaction"] + pairs["a"]

# Sanity guard: b, c, d must all be non-negative for a valid 2x2 table
invalid = (pairs["b"] < 0) | (pairs["c"] < 0) | (pairs["d"] < 0)
print(f"Pairs with an invalid contingency table (should be 0): {invalid.sum()}")
pairs = pairs[~invalid]

Pairs with an invalid contingency table (should be 0): 5990


In [ ]:
a, b, c, d = pairs["a"], pairs["b"], pairs["c"], pairs["d"]

pairs["PRR"] = (a / (a + b)) / (c / (c + d))
pairs["ROR"] = (a * d) / (b * c)

# Standard (non-Yates-corrected) chi-square for a 2x2 table
pairs["chi_square"] = N * (a * d - b * c) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))

pairs = pairs.replace([np.inf, -np.inf], np.nan)

In [ ]:
MIN_CASES = 3
PRR_THRESHOLD = 2
CHI2_THRESHOLD = 4

pairs["is_signal"] = (
    (pairs["a"] >= MIN_CASES) & (pairs["PRR"] >= PRR_THRESHOLD) & (pairs["chi_square"] >= CHI2_THRESHOLD)
)

n_signals = pairs["is_signal"].sum()
print(f"\nPairs meeting Evans criteria (PRR>={PRR_THRESHOLD}, chi2>={CHI2_THRESHOLD}, cases>={MIN_CASES}): "
      f"{n_signals:,} of {len(pairs):,} total pairs")

q7 = pairs[pairs["is_signal"]].sort_values("PRR", ascending=False).head(10)[
    ["drug_label", "pt_clean", "a", "PRR", "ROR", "chi_square"]
].rename(columns={"a": "co_occurrence_count", "pt_clean": "reaction"}).reset_index(drop=True)
q7[["PRR", "ROR", "chi_square"]] = q7[["PRR", "ROR", "chi_square"]].round(2)

print("\n" + "=" * 60)
print("Q7 — Top 10 disproportionate reporting signals (PRR/ROR)")
print("=" * 60)
print(q7.to_string(index=False))

question_results["q7_prr_ror_signals"] = q7
question_results["q7_excluded_outlier_reports"] = len(outlier_pids)
question_results["q7_total_pairs_evaluated"] = len(pairs)
question_results["q7_total_signals_flagged"] = int(n_signals)


Pairs meeting Evans criteria (PRR>=2, chi2>=4, cases>=3): 50,430 of 1,074,923 total pairs

Q7 — Top 10 disproportionate reporting signals (PRR/ROR)
                                         drug_label                                    reaction  co_occurrence_count        PRR        ROR  chi_square
                                          GUANIDINE                         AMYLOID ARTHROPATHY                    6 1605233.00        NaN    12326.01
                                     SULFAGUANIDINE                         AMYLOID ARTHROPATHY                    6 1375913.14 9631386.00    10563.45
                                       CACTINOMYCIN                  RHABDOMYOSARCOMA RECURRENT                    3  802618.00        NaN     8630.58
                    DICLOFENAC POTASSIUM\METAXALONE                     STOMATITIS HAEMORRHAGIC                    3  802618.00        NaN     8630.58
                                         AXATILIMAB                     HERPES SIMPLEX VIRAEMIA 

In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)
print(f"\nCached updated question_results (now includes all 13 questions) -> {cache_path}")


Cached updated question_results (now includes all 13 questions) -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 10 (FULL) — EXPORT EVERYTHING FOR MYSQL
# ------------------------------------------------------------------------------
# Purpose : Exports BOTH the 6 joined base tables (for writing your own SQL
#           analysis against) AND all 13 question_results tables (the
#           pandas-computed answers, useful for validating your SQL queries
#           against a known-correct result) to CSV, with auto-generated
#           CREATE TABLE + LOAD DATA INFILE statements for everything.
# Output  : /FAERS 2025/_cache/mysql_export/*.csv  (19 files)
#           /FAERS 2025/_cache/mysql_export/schema_and_load.sql
# ==============================================================================
import os
import pickle
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"
EXPORT_DIR = os.path.join(CACHE_DIR, "mysql_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

In [ ]:
# The 6 joined base tables — what you'll write your own SQL queries against
base_tables = {
    "report_level": load("report_level"),
    "drug_level": load("drug_level"),
    "reaction_level": load("reaction_level"),
    "rpsr_level": load("rpsr_level"),
    "drug_indi_level": load("drug_indi_level"),
    "drug_ther_level": load("drug_ther_level"),
}

In [ ]:
# The 13 question_results tables — only the actual dataframes; a few entries
# (q12_selected_drug, q7_excluded_outlier_reports, etc.) are plain scalars
# logged for reference, not result tables, and are skipped here.
question_results = load("question_results")
result_tables = {k: v for k, v in question_results.items() if isinstance(v, pd.DataFrame)}

all_tables = {**base_tables, **result_tables}
print(f"Exporting {len(base_tables)} base tables + {len(result_tables)} question-result tables "
      f"= {len(all_tables)} tables total\n")
for name, df in all_tables.items():
    print(f"{name:35s}: {df.shape}")

Exporting 6 base tables + 8 question-result tables = 14 tables total

report_level                       : (1617313, 15)
drug_level                         : (7469819, 22)
reaction_level                     : (5587113, 17)
rpsr_level                         : (43882, 17)
drug_indi_level                    : (4818036, 5)
drug_ther_level                    : (1956680, 8)
q1_age_gender_distribution         : (21, 4)
q2_reporting_lag_to_fda            : (5, 2)
q3_reporting_lag_to_manufacturer   : (4, 2)
q4_country_concentration           : (5, 3)
q5_top_drugs_pareto                : (10, 3)
q6_top_drugs_serious_outcome_pct   : (10, 4)
q13_dechal_rechal_top_pairs        : (10, 5)
q7_prr_ror_signals                 : (10, 6)


In [ ]:
for name, df in all_tables.items():
    csv_path = os.path.join(EXPORT_DIR, f"{name}.csv")
    export_df = df.copy()
    for col in export_df.select_dtypes(include="category").columns:
        export_df[col] = export_df[col].astype(str)
    export_df.to_csv(csv_path, index=False)
    size_mb = os.path.getsize(csv_path) / 1e6
    print(f"Exported {name} -> {csv_path} ({size_mb:,.1f} MB)")

Exported report_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/report_level.csv (140.0 MB)
Exported drug_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_level.csv (1,073.6 MB)
Exported reaction_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/reaction_level.csv (637.9 MB)
Exported rpsr_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/rpsr_level.csv (4.3 MB)
Exported drug_indi_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_indi_level.csv (291.9 MB)
Exported drug_ther_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_ther_level.csv (150.7 MB)
Exported q1_age_gender_distribution -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q1_age_gender_distribution.csv (0.0 MB)
Exported q2_reporting_lag_to_fda -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q2_reporting_lag_to_fda.csv (0.0 MB)
Exported q3_reporting_lag_to_manufacturer -> /content/drive/MyDrive/FAERS 2025/_cache/mysq

In [ ]:
def pandas_dtype_to_mysql(series: pd.Series) -> str:
    dtype = str(series.dtype)
    if "datetime" in dtype:
        return "DATE"
    if dtype in ("int64", "int32"):
        return "BIGINT"
    if dtype in ("float64", "float32"):
        return "DOUBLE"
    if dtype == "bool":
        return "BOOLEAN"
    max_len = series.astype(str).str.len().max()
    max_len = 50 if pd.isna(max_len) else int(max_len)
    if max_len <= 255:
        return f"VARCHAR({max(max_len + 20, 20)})"
    return "TEXT"

def generate_create_table_sql(df: pd.DataFrame, table_name: str) -> str:
    lines = [f"CREATE TABLE IF NOT EXISTS {table_name} ("]
    col_defs = [f"    `{col}` {pandas_dtype_to_mysql(df[col])}" for col in df.columns]
    lines.append(",\n".join(col_defs))
    lines.append(");")
    return "\n".join(lines)

def generate_load_data_sql(table_name: str, csv_filename: str, columns: list) -> str:
    col_list = ", ".join(f"`{c}`" for c in columns)
    return (
        f"LOAD DATA LOCAL INFILE '{csv_filename}'\n"
        f"INTO TABLE {table_name}\n"
        f"FIELDS TERMINATED BY ',' ENCLOSED BY '\"'\n"
        f"LINES TERMINATED BY '\\n'\n"
        f"IGNORE 1 ROWS\n"
        f"({col_list});"
    )

In [ ]:
sql_path = os.path.join(EXPORT_DIR, "schema_and_load.sql")
with open(sql_path, "w") as f:
    f.write("-- ============================================================\n")
    f.write("-- FAERS 2025 ANALYSIS — AUTO-GENERATED SCHEMA + BULK LOAD SCRIPT\n")
    f.write("-- ============================================================\n")
    f.write("-- CREATE DATABASE IF NOT EXISTS faers_2025;\n-- USE faers_2025;\n\n")
    f.write("-- If LOAD DATA LOCAL INFILE is disabled on your server:\n")
    f.write("-- SET GLOBAL local_infile = 1;   (reconnect the client after this)\n\n")
    f.write("-- Recommended session tuning before running the heavy JOIN queries\n")
    f.write("-- (Q7/Q13's drug-reaction pairwise queries in sql_analysis_queries.sql):\n")
    f.write("-- SET SESSION sort_buffer_size = 67108864;\n")
    f.write("-- SET SESSION tmp_table_size = 268435456;\n")
    f.write("-- SET SESSION max_heap_table_size = 268435456;\n\n")

    f.write("-- ==================== BASE TABLES ====================\n\n")
    for name, df in base_tables.items():
        f.write(f"-- ---- {name} ----\n")
        f.write(generate_create_table_sql(df, name) + "\n\n")
        f.write(generate_load_data_sql(name, f"{name}.csv", list(df.columns)) + "\n\n")

    f.write("-- ==================== QUESTION RESULT TABLES ====================\n\n")
    for name, df in result_tables.items():
        f.write(f"-- ---- {name} ----\n")
        f.write(generate_create_table_sql(df, name) + "\n\n")
        f.write(generate_load_data_sql(name, f"{name}.csv", list(df.columns)) + "\n\n")

    f.write("-- ==================== INDEXES (for the JOIN-heavy queries) ====================\n")
    f.write("-- These matter most for Q6, Q7, Q9, Q13 — anything joining drug_level or\n")
    f.write("-- reaction_level on primaryid, or grouping by drug_label/pt_clean.\n")
    f.write("CREATE INDEX idx_drug_primaryid ON drug_level(primaryid);\n")
    f.write("CREATE INDEX idx_drug_label_role ON drug_level(drug_label, role_cod);\n")
    f.write("CREATE INDEX idx_reaction_primaryid ON reaction_level(primaryid);\n")
    f.write("CREATE INDEX idx_reaction_pt ON reaction_level(pt_clean);\n")
    f.write("CREATE INDEX idx_ther_primaryid ON drug_ther_level(primaryid);\n")
    f.write("CREATE INDEX idx_indi_primaryid ON drug_indi_level(primaryid);\n")

print(f"\nSchema + load script written -> {sql_path}")
print("Download the mysql_export folder from Drive, edit the LOAD DATA LOCAL INFILE "
      "paths to your local CSV location, then run schema_and_load.sql in MySQL Workbench.")

NameError: name 'EXPORT_DIR' is not defined

In [ ]:
# ==============================================================================
# RECOMPUTE Q8-Q12 — these were lost when the Q13 session crash happened
# before question_results.pkl got saved. Q1-Q7 and Q13 are already safe in
# the pickle; this just fills in the gap.
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
print("Currently in question_results:")
for k in question_results:
    print(f"  - {k}")

report_level = load("report_level")
drug_level = load("drug_level")
reaction_level = load("reaction_level")
drug_indi_level = load("drug_indi_level")
drug_ther_level = load("drug_ther_level")

top10_drugs = question_results["q5_top_drugs_pareto"]["drug_label"].tolist()
print(f"\nTop 10 suspect drugs from Q5: {top10_drugs}")

Currently in question_results:
  - q1_age_gender_distribution
  - q2_reporting_lag_to_fda
  - q3_reporting_lag_to_manufacturer
  - q4_country_concentration
  - q5_top_drugs_pareto
  - q6_top_drugs_serious_outcome_pct
  - q13_dechal_rechal_top_pairs
  - q13_excluded_outlier_reports
  - q7_prr_ror_signals
  - q7_excluded_outlier_reports
  - q7_total_pairs_evaluated
  - q7_total_signals_flagged
  - q8_top_indication_per_drug
  - q9_top_reactions_serious_outcome
  - q10_country_death_rate
  - q11_reporter_source_vs_severity
  - q12_duration_vs_severity
  - q12_selected_drug

Top 10 suspect drugs from Q5: ['TIRZEPATIDE', 'DUPILUMAB', 'INFLIXIMAB-DYYB', 'RITUXIMAB', 'METHOTREXATE', 'INFLIXIMAB', 'TOCILIZUMAB', 'PREDNISONE', 'ADALIMUMAB', 'OMALIZUMAB']


In [ ]:
# ==============================================================================
# Q8 — Most common indication for the top drugs
# ==============================================================================
q8_rows = []
for d in top10_drugs:
    sub = drug_indi_level[drug_indi_level["drug_label"] == d]
    total_entries = len(sub)
    placeholder_pct = round(sub["is_unknown_indication"].mean() * 100, 2) if total_entries else np.nan
    real = sub[~sub["is_unknown_indication"]]
    if len(real):
        vc = real["indi_pt_clean"].value_counts()
        top_indi, top_indi_count = vc.idxmax(), vc.max()
    else:
        top_indi, top_indi_count = "N/A", 0
    q8_rows.append({
        "drug_label": d, "total_indication_entries": total_entries,
        "unknown_indication_pct": placeholder_pct,
        "top_real_indication": top_indi, "top_indication_report_count": top_indi_count,
    })
q8 = pd.DataFrame(q8_rows)
print("\nQ8:")
print(q8.to_string(index=False))
question_results["q8_top_indication_per_drug"] = q8


Q8:
     drug_label  total_indication_entries  unknown_indication_pct  top_real_indication  top_indication_report_count
    TIRZEPATIDE                    199989                   49.32       WEIGHT CONTROL                        50643
      DUPILUMAB                    157252                    1.19    DERMATITIS ATOPIC                        64855
INFLIXIMAB-DYYB                      9111                    5.13      CROHN'S DISEASE                         4793
      RITUXIMAB                     36539                   10.85 RHEUMATOID ARTHRITIS                        10658
   METHOTREXATE                     34625                   25.42 RHEUMATOID ARTHRITIS                        12991
     INFLIXIMAB                     26370                   14.61 RHEUMATOID ARTHRITIS                         7667
    TOCILIZUMAB                     21912                   16.99 RHEUMATOID ARTHRITIS                        14008
     PREDNISONE                     55893                   30.87 R

In [ ]:
# ==============================================================================
# Q9 — Top 10 reactions linked to serious outcomes
# ==============================================================================
serious_reac = reaction_level[reaction_level["has_serious_outcome"]]
vc = serious_reac["pt_clean"].value_counts()
total = vc.sum()
top10_reac = vc.head(10)
q9 = pd.DataFrame({
    "reaction": top10_reac.index, "count": top10_reac.values,
    "pct_of_serious_reaction_listings": (top10_reac / total * 100).round(2).values,
})
print("\nQ9:")
print(q9.to_string(index=False))
question_results["q9_top_reactions_serious_outcome"] = q9


Q9:
        reaction  count  pct_of_serious_reaction_listings
   OFF LABEL USE  74167                              1.89
           DEATH  59696                              1.52
DRUG INEFFECTIVE  46753                              1.19
         FATIGUE  43675                              1.11
       DIARRHOEA  39328                              1.00
          NAUSEA  38773                              0.99
       PNEUMONIA  32456                              0.83
        DYSPNOEA  31763                              0.81
        VOMITING  30073                              0.77
            PAIN  29866                              0.76


In [ ]:
# ==============================================================================
# Q10 — Country death rate (min 100 reports)
# ==============================================================================
MIN_REPORTS_Q10 = 100
country = report_level.copy()
country["reporter_country"] = country["reporter_country"].astype(str).str.strip().str.upper()
country = country[~country["reporter_country"].isin(["", "NAN"])]
q10_full = country.groupby("reporter_country").agg(
    total_reports=("primaryid", "size"), death_reports=("has_death", "sum"),
).reset_index()
q10_full["death_rate_pct"] = round(q10_full["death_reports"] / q10_full["total_reports"] * 100, 2)
q10 = q10_full[q10_full["total_reports"] >= MIN_REPORTS_Q10].sort_values(
    "death_rate_pct", ascending=False
).head(10).reset_index(drop=True)
print("\nQ10:")
print(q10.to_string(index=False))
question_results["q10_country_death_rate"] = q10


Q10:
reporter_country  total_reports  death_reports  death_rate_pct
              KE            158             98           62.03
              MA            710            399           56.20
              NP            274            149           54.38
              ID            839            399           47.56
              PK           1491            677           45.41
              PH            860            382           44.42
              BY            114             40           35.09
              HK            409            142           34.72
              VN           1042            354           33.97
              DZ            175             58           33.14


In [ ]:
# ==============================================================================
# Q11 — Reporter source vs. severity
# ==============================================================================
rpsr_level = load("rpsr_level")
q11 = rpsr_level.groupby("rpsr_cod_clean").agg(
    total_reports=("primaryid", "size"),
    serious_reports=("has_serious_outcome", "sum"),
    death_reports=("has_death", "sum"),
).reset_index()
q11["serious_pct"] = round(q11["serious_reports"] / q11["total_reports"] * 100, 2)
q11["death_pct"] = round(q11["death_reports"] / q11["total_reports"] * 100, 2)
q11 = q11.sort_values("total_reports", ascending=False)
print("\nQ11:")
print(q11.to_string(index=False))
question_results["q11_reporter_source_vs_severity"] = q11


Q11:
rpsr_cod_clean  total_reports  serious_reports  death_reports  serious_pct  death_pct
           CSM          24282            12958           1284        53.36       5.29
            HP          19132            13048           2809        68.20      14.68
           FGN            468              407             10        86.97       2.14


In [ ]:
# ==============================================================================
# Q12 — Therapy duration vs. severity for Q5's #1 drug
# ==============================================================================
SELECTED_DRUG = top10_drugs[0]
ther_scope = drug_ther_level[
    (drug_ther_level["drug_label"] == SELECTED_DRUG) &
    (drug_ther_level["role_cod"].isin(["PS", "SS"]))
].merge(report_level[["primaryid", "has_serious_outcome"]], on="primaryid", how="left")
ther_scope = ther_scope[ther_scope["duration_days"].notna()]

DURATION_BINS = [0, 7, 30, 90, 180, 365, 1_000_000]
DURATION_LABELS = ["<=7d", "8-30d", "31-90d", "91-180d", "181-365d", ">365d"]
ther_scope["duration_bucket"] = pd.cut(
    ther_scope["duration_days"], bins=DURATION_BINS, labels=DURATION_LABELS, include_lowest=True
)
q12 = ther_scope.groupby("duration_bucket", observed=True).agg(
    n_reports=("primaryid", "size"),
    serious_pct=("has_serious_outcome", lambda s: round(s.mean() * 100, 2)),
).reset_index()
print(f"\nQ12 (drug: {SELECTED_DRUG}):")
print(q12.to_string(index=False))
question_results["q12_duration_vs_severity"] = q12
question_results["q12_selected_drug"] = SELECTED_DRUG


Q12 (drug: TIRZEPATIDE):
duration_bucket  n_reports  serious_pct
           <=7d        719        64.39
          8-30d       1981        67.54
         31-90d       2337        76.77
        91-180d       1244        76.29
       181-365d        878        76.08
          >365d        427        75.64


In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)

print("\n" + "=" * 60)
print(f"Fixed. question_results now has {len(question_results)} entries -> {cache_path}")
print("=" * 60)
for k in question_results:
    print(f"  - {k}")


Fixed. question_results now has 18 entries -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl
  - q1_age_gender_distribution
  - q2_reporting_lag_to_fda
  - q3_reporting_lag_to_manufacturer
  - q4_country_concentration
  - q5_top_drugs_pareto
  - q6_top_drugs_serious_outcome_pct
  - q13_dechal_rechal_top_pairs
  - q13_excluded_outlier_reports
  - q7_prr_ror_signals
  - q7_excluded_outlier_reports
  - q7_total_pairs_evaluated
  - q7_total_signals_flagged
  - q8_top_indication_per_drug
  - q9_top_reactions_serious_outcome
  - q10_country_death_rate
  - q11_reporter_source_vs_severity
  - q12_duration_vs_severity
  - q12_selected_drug


In [ ]:
# ==============================================================================
# FAERS 2025 (Q1-Q4) ADVERSE EVENT ANALYSIS PROJECT
# STEP 10 (FULL) — EXPORT EVERYTHING FOR MYSQL
# ------------------------------------------------------------------------------
# Purpose : Exports BOTH the 6 joined base tables (for writing your own SQL
#           analysis against) AND all 13 question_results tables (the
#           pandas-computed answers, useful for validating your SQL queries
#           against a known-correct result) to CSV, with auto-generated
#           CREATE TABLE + LOAD DATA INFILE statements for everything.
# Output  : /FAERS 2025/_cache/mysql_export/*.csv  (19 files)
#           /FAERS 2025/_cache/mysql_export/schema_and_load.sql
# ==============================================================================
import os
import pickle
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"
EXPORT_DIR = os.path.join(CACHE_DIR, "mysql_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

# The 6 joined base tables — what you'll write your own SQL queries against
base_tables = {
    "report_level": load("report_level"),
    "drug_level": load("drug_level"),
    "reaction_level": load("reaction_level"),
    "rpsr_level": load("rpsr_level"),
    "drug_indi_level": load("drug_indi_level"),
    "drug_ther_level": load("drug_ther_level"),
}

# The 13 question_results tables — only the actual dataframes; a few entries
# (q12_selected_drug, q7_excluded_outlier_reports, etc.) are plain scalars
# logged for reference, not result tables, and are skipped here.
question_results = load("question_results")
result_tables = {k: v for k, v in question_results.items() if isinstance(v, pd.DataFrame)}

all_tables = {**base_tables, **result_tables}
print(f"Exporting {len(base_tables)} base tables + {len(result_tables)} question-result tables "
      f"= {len(all_tables)} tables total\n")
for name, df in all_tables.items():
    print(f"{name:35s}: {df.shape}")

Exporting 6 base tables + 13 question-result tables = 19 tables total

report_level                       : (1617313, 15)
drug_level                         : (7469819, 22)
reaction_level                     : (5587113, 17)
rpsr_level                         : (43882, 17)
drug_indi_level                    : (4818036, 5)
drug_ther_level                    : (1956680, 8)
q1_age_gender_distribution         : (21, 4)
q2_reporting_lag_to_fda            : (5, 2)
q3_reporting_lag_to_manufacturer   : (4, 2)
q4_country_concentration           : (5, 3)
q5_top_drugs_pareto                : (10, 3)
q6_top_drugs_serious_outcome_pct   : (10, 4)
q13_dechal_rechal_top_pairs        : (10, 5)
q7_prr_ror_signals                 : (10, 6)
q8_top_indication_per_drug         : (10, 5)
q9_top_reactions_serious_outcome   : (10, 3)
q10_country_death_rate             : (10, 4)
q11_reporter_source_vs_severity    : (3, 6)
q12_duration_vs_severity           : (6, 3)


In [ ]:
for name, df in all_tables.items():
    csv_path = os.path.join(EXPORT_DIR, f"{name}.csv")
    export_df = df.copy()
    for col in export_df.select_dtypes(include="category").columns:
        export_df[col] = export_df[col].astype(str)
    export_df.to_csv(csv_path, index=False)
    size_mb = os.path.getsize(csv_path) / 1e6
    print(f"Exported {name} -> {csv_path} ({size_mb:,.1f} MB)")

Exported report_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/report_level.csv (140.0 MB)
Exported drug_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_level.csv (1,073.6 MB)
Exported reaction_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/reaction_level.csv (637.9 MB)
Exported rpsr_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/rpsr_level.csv (4.3 MB)
Exported drug_indi_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_indi_level.csv (291.9 MB)
Exported drug_ther_level -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/drug_ther_level.csv (150.7 MB)
Exported q1_age_gender_distribution -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q1_age_gender_distribution.csv (0.0 MB)
Exported q2_reporting_lag_to_fda -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q2_reporting_lag_to_fda.csv (0.0 MB)
Exported q3_reporting_lag_to_manufacturer -> /content/drive/MyDrive/FAERS 2025/_cache/mysq

In [ ]:
def pandas_dtype_to_mysql(series: pd.Series) -> str:
    dtype = str(series.dtype)
    if "datetime" in dtype:
        return "DATE"
    if dtype in ("int64", "int32"):
        return "BIGINT"
    if dtype in ("float64", "float32"):
        return "DOUBLE"
    if dtype == "bool":
        return "BOOLEAN"
    max_len = series.astype(str).str.len().max()
    max_len = 50 if pd.isna(max_len) else int(max_len)
    if max_len <= 255:
        return f"VARCHAR({max(max_len + 20, 20)})"
    return "TEXT"

def generate_create_table_sql(df: pd.DataFrame, table_name: str) -> str:
    lines = [f"CREATE TABLE IF NOT EXISTS {table_name} ("]
    col_defs = [f"    `{col}` {pandas_dtype_to_mysql(df[col])}" for col in df.columns]
    lines.append(",\n".join(col_defs))
    lines.append(");")
    return "\n".join(lines)

def generate_load_data_sql(table_name: str, csv_filename: str, columns: list) -> str:
    col_list = ", ".join(f"`{c}`" for c in columns)
    return (
        f"LOAD DATA LOCAL INFILE '{csv_filename}'\n"
        f"INTO TABLE {table_name}\n"
        f"FIELDS TERMINATED BY ',' ENCLOSED BY '\"'\n"
        f"LINES TERMINATED BY '\\n'\n"
        f"IGNORE 1 ROWS\n"
        f"({col_list});"
    )

In [ ]:
sql_path = os.path.join(EXPORT_DIR, "schema_and_load.sql")
with open(sql_path, "w") as f:
    f.write("-- ============================================================\n")
    f.write("-- FAERS 2025 ANALYSIS — AUTO-GENERATED SCHEMA + BULK LOAD SCRIPT\n")
    f.write("-- ============================================================\n")
    f.write("-- CREATE DATABASE IF NOT EXISTS faers_2025;\n-- USE faers_2025;\n\n")
    f.write("-- If LOAD DATA LOCAL INFILE is disabled on your server:\n")
    f.write("-- SET GLOBAL local_infile = 1;   (reconnect the client after this)\n\n")
    f.write("-- Recommended session tuning before running the heavy JOIN queries\n")
    f.write("-- (Q7/Q13's drug-reaction pairwise queries in sql_analysis_queries.sql):\n")
    f.write("-- SET SESSION sort_buffer_size = 67108864;\n")
    f.write("-- SET SESSION tmp_table_size = 268435456;\n")
    f.write("-- SET SESSION max_heap_table_size = 268435456;\n\n")

    f.write("-- ==================== BASE TABLES ====================\n\n")
    for name, df in base_tables.items():
        f.write(f"-- ---- {name} ----\n")
        f.write(generate_create_table_sql(df, name) + "\n\n")
        f.write(generate_load_data_sql(name, f"{name}.csv", list(df.columns)) + "\n\n")

    f.write("-- ==================== QUESTION RESULT TABLES ====================\n\n")
    for name, df in result_tables.items():
        f.write(f"-- ---- {name} ----\n")
        f.write(generate_create_table_sql(df, name) + "\n\n")
        f.write(generate_load_data_sql(name, f"{name}.csv", list(df.columns)) + "\n\n")

    f.write("-- ==================== INDEXES (for the JOIN-heavy queries) ====================\n")
    f.write("-- These matter most for Q6, Q7, Q9, Q13 — anything joining drug_level or\n")
    f.write("-- reaction_level on primaryid, or grouping by drug_label/pt_clean.\n")
    f.write("CREATE INDEX idx_drug_primaryid ON drug_level(primaryid);\n")
    f.write("CREATE INDEX idx_drug_label_role ON drug_level(drug_label, role_cod);\n")
    f.write("CREATE INDEX idx_reaction_primaryid ON reaction_level(primaryid);\n")
    f.write("CREATE INDEX idx_reaction_pt ON reaction_level(pt_clean);\n")
    f.write("CREATE INDEX idx_ther_primaryid ON drug_ther_level(primaryid);\n")
    f.write("CREATE INDEX idx_indi_primaryid ON drug_indi_level(primaryid);\n")

print(f"\nSchema + load script written -> {sql_path}")
print("Download the mysql_export folder from Drive, edit the LOAD DATA LOCAL INFILE "
      "paths to your local CSV location, then run schema_and_load.sql in MySQL Workbench.")


Schema + load script written -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/schema_and_load.sql
Download the mysql_export folder from Drive, edit the LOAD DATA LOCAL INFILE paths to your local CSV location, then run schema_and_load.sql in MySQL Workbench.


In [ ]:
# ==============================================================================
# FIX Q6, Q7, Q13 — all three share the same root cause: duplicate
# (primaryid, drug_label) rows in drug_level inflating counts. Q6's
# serious_pct exceeding 100% (confirmed via the MySQL data preview) proves
# this bug exists in the ORIGINAL Python computation too, not just the SQL
# side — so all three need recomputing here, not just Q7/Q13.
# ==============================================================================
import os
import gc
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
drug_level = load("drug_level")
reaction_level = load("reaction_level")
report_level = load("report_level")

top10_drugs = question_results["q5_top_drugs_pareto"]["drug_label"].tolist()
print(f"Top 10 suspect drugs from Q5: {top10_drugs}")

Top 10 suspect drugs from Q5: ['TIRZEPATIDE', 'DUPILUMAB', 'INFLIXIMAB-DYYB', 'RITUXIMAB', 'METHOTREXATE', 'INFLIXIMAB', 'TOCILIZUMAB', 'PREDNISONE', 'ADALIMUMAB', 'OMALIZUMAB']


In [ ]:
suspect_raw = drug_level[drug_level["role_cod"].isin(["PS", "SS"])]
before = len(suspect_raw)

suspect_dedup = suspect_raw.drop_duplicates(subset=["primaryid", "drug_label"]).copy()
after = len(suspect_dedup)
print(f"Deduplicated suspect drug rows: {before:,} -> {after:,} "
      f"({before - after:,} duplicate (primaryid, drug_label) rows removed)")

# ==============================================================================
# Q6 FIX — serious outcome % per top drug
# ==============================================================================
top10_subset = suspect_dedup[suspect_dedup["drug_label"].isin(top10_drugs)]

q6 = (
    top10_subset.groupby("drug_label", observed=True)
    .agg(total_reports=("primaryid", "nunique"),
         serious_reports=("has_serious_outcome", "sum"))
    .reset_index()
)
q6["serious_pct"] = round(q6["serious_reports"] / q6["total_reports"] * 100, 2)
q6 = q6.sort_values("total_reports", ascending=False)

print("\nQ6 (corrected) — all serious_pct values should now be <= 100:")
print(q6.to_string(index=False))
assert (q6["serious_pct"] <= 100).all(), "Still seeing >100% — investigate before trusting this"
question_results["q6_top_drugs_serious_outcome_pct"] = q6

Deduplicated suspect drug rows: 4,603,624 -> 2,635,125 (1,968,499 duplicate (primaryid, drug_label) rows removed)

Q6 (corrected) — all serious_pct values should now be <= 100:
     drug_label  total_reports  serious_reports  serious_pct
      DUPILUMAB         143867            14383        10.00
    TIRZEPATIDE          60763            12489        20.55
     ADALIMUMAB          24112            19403        80.47
     PREDNISONE          23948            22595        94.35
      RITUXIMAB          23574            22702        96.30
   METHOTREXATE          17615            16925        96.08
     INFLIXIMAB          14920            14278        95.70
     OMALIZUMAB          10841             5270        48.61
    TOCILIZUMAB          10043             7872        78.38
INFLIXIMAB-DYYB           8368             7656        91.49


In [ ]:
suspect_small = suspect_raw.groupby(["primaryid", "drug_label"], observed=True).agg(
    dechal_clean=("dechal_clean", lambda s: "Y" if (s == "Y").any() else s.iloc[0]),
    rechal_clean=("rechal_clean", lambda s: "Y" if (s == "Y").any() else s.iloc[0]),
).reset_index()

reac_small = reaction_level[["primaryid", "pt_clean"]].copy()

In [ ]:
drug_counts = suspect_small.groupby("primaryid").size()
reac_counts = reac_small.groupby("primaryid").size()
all_pids = drug_counts.index.union(reac_counts.index)
combo_size = (drug_counts.reindex(all_pids, fill_value=0) * reac_counts.reindex(all_pids, fill_value=0))

FANOUT_CAP = 400
outlier_pids = combo_size[combo_size > FANOUT_CAP].index
print(f"\nReports excluded for exceeding a {FANOUT_CAP}-row fan-out cap: {len(outlier_pids):,}")

suspect_small = suspect_small[~suspect_small["primaryid"].isin(outlier_pids)]
reac_small = reac_small[~reac_small["primaryid"].isin(outlier_pids)]
del drug_counts, reac_counts, all_pids, combo_size
gc.collect()


Reports excluded for exceeding a 400-row fan-out cap: 6,444


14

In [ ]:
n_drug = suspect_small.groupby("drug_label", observed=True)["primaryid"].nunique()
n_reaction = reac_small.groupby("pt_clean", observed=True)["primaryid"].nunique()
N = len(set(suspect_small["primaryid"]).union(set(reac_small["primaryid"])))
print(f"N = {N:,} | {len(n_drug):,} distinct suspect drugs | {len(n_reaction):,} distinct reactions")

N = 1,610,869 | 6,547 distinct suspect drugs | 16,775 distinct reactions


In [ ]:
BATCH_SIZE = 50_000
unique_pids = suspect_small["primaryid"].unique()
n_batches = (len(unique_pids) // BATCH_SIZE) + 1
print(f"Processing {len(unique_pids):,} primaryids in {n_batches} batches")

pair_accumulator = {}

for i in range(0, len(unique_pids), BATCH_SIZE):
    batch_ids = set(unique_pids[i:i + BATCH_SIZE])
    drug_chunk = suspect_small[suspect_small["primaryid"].isin(batch_ids)]
    reac_chunk = reac_small[reac_small["primaryid"].isin(batch_ids)]
    merged_chunk = drug_chunk.merge(reac_chunk, on="primaryid", how="inner")

    if len(merged_chunk):
        grp = merged_chunk.groupby(["drug_label", "pt_clean"], observed=True).agg(
            pair_count=("primaryid", "size"),
            dechal_pos=("dechal_clean", lambda s: (s == "Y").sum()),
            rechal_pos=("rechal_clean", lambda s: (s == "Y").sum()),
        ).reset_index()
        for row in grp.itertuples(index=False):
            key = (row.drug_label, row.pt_clean)
            if key not in pair_accumulator:
                pair_accumulator[key] = [0, 0, 0]
            pair_accumulator[key][0] += row.pair_count
            pair_accumulator[key][1] += row.dechal_pos
            pair_accumulator[key][2] += row.rechal_pos

    del drug_chunk, reac_chunk, merged_chunk
    gc.collect()
    print(f"  batch {i // BATCH_SIZE + 1}/{n_batches} done, {len(pair_accumulator):,} distinct pairs so far")

pairs = pd.DataFrame(
    [(d, r, v[0], v[1], v[2]) for (d, r), v in pair_accumulator.items()],
    columns=["drug_label", "pt_clean", "a", "dechal_pos", "rechal_pos"]
)
print(f"\nTotal distinct drug-reaction pairs: {len(pairs):,}")

Processing 1,610,859 primaryids in 33 batches
  batch 1/33 done, 273,091 distinct pairs so far
  batch 2/33 done, 379,645 distinct pairs so far
  batch 3/33 done, 451,774 distinct pairs so far
  batch 4/33 done, 501,708 distinct pairs so far
  batch 5/33 done, 535,816 distinct pairs so far
  batch 6/33 done, 567,903 distinct pairs so far
  batch 7/33 done, 595,317 distinct pairs so far
  batch 8/33 done, 621,631 distinct pairs so far
  batch 9/33 done, 649,026 distinct pairs so far
  batch 10/33 done, 672,366 distinct pairs so far
  batch 11/33 done, 695,744 distinct pairs so far
  batch 12/33 done, 719,896 distinct pairs so far
  batch 13/33 done, 744,086 distinct pairs so far
  batch 14/33 done, 769,693 distinct pairs so far
  batch 15/33 done, 788,694 distinct pairs so far
  batch 16/33 done, 809,506 distinct pairs so far
  batch 17/33 done, 830,376 distinct pairs so far
  batch 18/33 done, 849,728 distinct pairs so far
  batch 19/33 done, 869,873 distinct pairs so far
  batch 20/33

In [ ]:
pairs["n_drug"] = pairs["drug_label"].map(n_drug)
pairs["n_reaction"] = pairs["pt_clean"].map(n_reaction)
impossible = pairs["a"] > pairs["n_drug"]
print(f"Pairs where a > n_drug (should be 0): {impossible.sum()}")
assert impossible.sum() == 0, "Still impossible values — investigate before trusting results"

# ==============================================================================
# Q7 FIX — PRR/ROR
# ==============================================================================
pairs["b"] = pairs["n_drug"] - pairs["a"]
pairs["c"] = pairs["n_reaction"] - pairs["a"]
pairs["d"] = N - pairs["n_drug"] - pairs["n_reaction"] + pairs["a"]

invalid = (pairs["b"] < 0) | (pairs["c"] < 0) | (pairs["d"] < 0)
pairs_valid = pairs[~invalid].copy()

a, b, c, d = pairs_valid["a"], pairs_valid["b"], pairs_valid["c"], pairs_valid["d"]
pairs_valid["PRR"] = (a / (a + b)) / (c / (c + d))
pairs_valid["ROR"] = (a * d) / (b * c)
pairs_valid["chi_square"] = N * (a * d - b * c) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
pairs_valid = pairs_valid.replace([np.inf, -np.inf], np.nan)

MIN_CASES, PRR_THRESHOLD, CHI2_THRESHOLD = 3, 2, 4
pairs_valid["is_signal"] = (
    (pairs_valid["a"] >= MIN_CASES) & (pairs_valid["PRR"] >= PRR_THRESHOLD) & (pairs_valid["chi_square"] >= CHI2_THRESHOLD)
)
n_signals = pairs_valid["is_signal"].sum()
print(f"\nPairs meeting Evans criteria: {n_signals:,} of {len(pairs_valid):,}")

q7 = pairs_valid[pairs_valid["is_signal"]].sort_values("PRR", ascending=False).head(10)[
    ["drug_label", "pt_clean", "a", "PRR", "ROR", "chi_square"]
].rename(columns={"a": "co_occurrence_count", "pt_clean": "reaction"}).reset_index(drop=True)
q7[["PRR", "ROR", "chi_square"]] = q7[["PRR", "ROR", "chi_square"]].round(2)

print("\nQ7 (corrected):")
print(q7.to_string(index=False))
question_results["q7_prr_ror_signals"] = q7

# ==============================================================================
# Q13 FIX — dechal/rechal
# ==============================================================================
MIN_PAIR_COUNT_Q13 = 20
pairs["pct_dechal_positive"] = round(pairs["dechal_pos"] / pairs["a"] * 100, 2)
pairs["pct_rechal_positive"] = round(pairs["rechal_pos"] / pairs["a"] * 100, 2)

q13 = pairs[pairs["a"] >= MIN_PAIR_COUNT_Q13].sort_values("a", ascending=False).head(10)[
    ["drug_label", "pt_clean", "a", "pct_dechal_positive", "pct_rechal_positive"]
].rename(columns={"a": "pair_count"}).reset_index(drop=True)

print("\nQ13 (corrected):")
print(q13.to_string(index=False))
question_results["q13_dechal_rechal_top_pairs"] = q13

Pairs where a > n_drug (should be 0): 0

Pairs meeting Evans criteria: 39,103 of 1,117,569

Q7 (corrected):
                                         drug_label                                    reaction  co_occurrence_count        PRR        ROR  chi_square
                                  PHENOL\ZINC OXIDE                        BILE OUTPUT ABNORMAL                    6 1610863.00        NaN    26666.64
                                       CACTINOMYCIN                  RHABDOMYOSARCOMA RECURRENT                    3  805433.00        NaN    18668.99
                               DELAVIRDINE MESYLATE        PROGRESSIVE EXTERNAL OPHTHALMOPLEGIA                    3  805433.00        NaN    18668.99
                                          DEMANNOSE                           HYPOOSMOLAR STATE                    3  690369.43 1208145.75     9998.48
                                       CAPLACIZUMAB                            PLACENTAL OEDEMA                    7  593471.05  939661.9

In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)
print(f"\nCached updated question_results -> {cache_path}")


Cached updated question_results -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl


In [ ]:
EXPORT_DIR = os.path.join(CACHE_DIR, "mysql_export")
os.makedirs(EXPORT_DIR, exist_ok=True)

for name, df in [("q6_top_drugs_serious_outcome_pct", q6),
                  ("q7_prr_ror_signals", q7),
                  ("q13_dechal_rechal_top_pairs", q13)]:
    csv_path = os.path.join(EXPORT_DIR, f"{name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"Exported corrected {name} -> {csv_path}")

print("\nDone. All 3 corrected CSVs are in the mysql_export folder in Drive.")

Exported corrected q6_top_drugs_serious_outcome_pct -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q6_top_drugs_serious_outcome_pct.csv
Exported corrected q7_prr_ror_signals -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q7_prr_ror_signals.csv
Exported corrected q13_dechal_rechal_top_pairs -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q13_dechal_rechal_top_pairs.csv

Done. All 3 corrected CSVs are in the mysql_export folder in Drive.


In [ ]:
# ==============================================================================
# FIX Q7 — raise minimum case count from 3 to 20 for the RANKING specifically.
# The bare Evans minimum (3 cases) is technically valid but produces a "top
# signal" driven by a 3-report pair with a PRR in the millions — a
# mathematical artifact of tiny denominators, not a meaningful finding.
# Raising to 20 (matching Q13's threshold, for consistency) keeps the
# ranking defensible. All 3 Evans criteria (PRR>=2, chi2>=4) are still
# applied on top of the higher case-count floor.
# ==============================================================================
import os
import gc
import pickle
import numpy as np
import pandas as pd

CACHE_DIR = "/content/drive/MyDrive/FAERS 2025/_cache"

def load(name):
    with open(os.path.join(CACHE_DIR, f"{name}.pkl"), "rb") as f:
        return pickle.load(f)

question_results = load("question_results")
drug_level = load("drug_level")
reaction_level = load("reaction_level")

In [ ]:
suspect_raw = drug_level[drug_level["role_cod"].isin(["PS", "SS"])]
suspect_small = suspect_raw.groupby(["primaryid", "drug_label"], observed=True).agg(
    dechal_clean=("dechal_clean", lambda s: "Y" if (s == "Y").any() else s.iloc[0]),
    rechal_clean=("rechal_clean", lambda s: "Y" if (s == "Y").any() else s.iloc[0]),
).reset_index()

reac_small = reaction_level[["primaryid", "pt_clean"]].copy()

In [ ]:
drug_counts = suspect_small.groupby("primaryid").size()
reac_counts = reac_small.groupby("primaryid").size()
all_pids = drug_counts.index.union(reac_counts.index)
combo_size = (drug_counts.reindex(all_pids, fill_value=0) * reac_counts.reindex(all_pids, fill_value=0))

FANOUT_CAP = 400
outlier_pids = combo_size[combo_size > FANOUT_CAP].index
print(f"Reports excluded for exceeding a {FANOUT_CAP}-row fan-out cap: {len(outlier_pids):,}")

suspect_small = suspect_small[~suspect_small["primaryid"].isin(outlier_pids)]
reac_small = reac_small[~reac_small["primaryid"].isin(outlier_pids)]
del drug_counts, reac_counts, all_pids, combo_size
gc.collect()

Reports excluded for exceeding a 400-row fan-out cap: 6,444


0

In [ ]:
n_drug = suspect_small.groupby("drug_label", observed=True)["primaryid"].nunique()
n_reaction = reac_small.groupby("pt_clean", observed=True)["primaryid"].nunique()
N = len(set(suspect_small["primaryid"]).union(set(reac_small["primaryid"])))
print(f"N = {N:,} | {len(n_drug):,} distinct suspect drugs | {len(n_reaction):,} distinct reactions")

N = 1,610,869 | 6,547 distinct suspect drugs | 16,775 distinct reactions


In [ ]:
BATCH_SIZE = 50_000
unique_pids = suspect_small["primaryid"].unique()
n_batches = (len(unique_pids) // BATCH_SIZE) + 1
print(f"Processing {len(unique_pids):,} primaryids in {n_batches} batches")

pair_accumulator = {}

for i in range(0, len(unique_pids), BATCH_SIZE):
    batch_ids = set(unique_pids[i:i + BATCH_SIZE])
    drug_chunk = suspect_small[suspect_small["primaryid"].isin(batch_ids)]
    reac_chunk = reac_small[reac_small["primaryid"].isin(batch_ids)]
    merged_chunk = drug_chunk.merge(reac_chunk, on="primaryid", how="inner")

    if len(merged_chunk):
        grp = merged_chunk.groupby(["drug_label", "pt_clean"], observed=True).size()
        for key, val in grp.items():
            pair_accumulator[key] = pair_accumulator.get(key, 0) + val

    del drug_chunk, reac_chunk, merged_chunk
    gc.collect()
    print(f"  batch {i // BATCH_SIZE + 1}/{n_batches} done, {len(pair_accumulator):,} distinct pairs so far")

pairs = pd.DataFrame(
    [(d, r, a) for (d, r), a in pair_accumulator.items()],
    columns=["drug_label", "pt_clean", "a"]
)
print(f"\nTotal distinct drug-reaction pairs: {len(pairs):,}")

Processing 1,610,859 primaryids in 33 batches
  batch 1/33 done, 273,091 distinct pairs so far
  batch 2/33 done, 379,645 distinct pairs so far
  batch 3/33 done, 451,774 distinct pairs so far
  batch 4/33 done, 501,708 distinct pairs so far
  batch 5/33 done, 535,816 distinct pairs so far
  batch 6/33 done, 567,903 distinct pairs so far
  batch 7/33 done, 595,317 distinct pairs so far
  batch 8/33 done, 621,631 distinct pairs so far
  batch 9/33 done, 649,026 distinct pairs so far
  batch 10/33 done, 672,366 distinct pairs so far
  batch 11/33 done, 695,744 distinct pairs so far
  batch 12/33 done, 719,896 distinct pairs so far
  batch 13/33 done, 744,086 distinct pairs so far
  batch 14/33 done, 769,693 distinct pairs so far
  batch 15/33 done, 788,694 distinct pairs so far
  batch 16/33 done, 809,506 distinct pairs so far
  batch 17/33 done, 830,376 distinct pairs so far
  batch 18/33 done, 849,728 distinct pairs so far
  batch 19/33 done, 869,873 distinct pairs so far
  batch 20/33

In [ ]:
pairs["n_drug"] = pairs["drug_label"].map(n_drug)
pairs["n_reaction"] = pairs["pt_clean"].map(n_reaction)

impossible = pairs["a"] > pairs["n_drug"]
assert impossible.sum() == 0, f"{impossible.sum()} impossible pairs found — dedup bug not actually fixed"

pairs["b"] = pairs["n_drug"] - pairs["a"]
pairs["c"] = pairs["n_reaction"] - pairs["a"]
pairs["d"] = N - pairs["n_drug"] - pairs["n_reaction"] + pairs["a"]

invalid = (pairs["b"] < 0) | (pairs["c"] < 0) | (pairs["d"] < 0)
pairs_valid = pairs[~invalid].copy()

a, b, c, d = pairs_valid["a"], pairs_valid["b"], pairs_valid["c"], pairs_valid["d"]
pairs_valid["PRR"] = (a / (a + b)) / (c / (c + d))
pairs_valid["ROR"] = (a * d) / (b * c)
pairs_valid["chi_square"] = N * (a * d - b * c) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
pairs_valid = pairs_valid.replace([np.inf, -np.inf], np.nan)

In [ ]:
MIN_CASES, PRR_THRESHOLD, CHI2_THRESHOLD = 20, 2, 4
pairs_valid["is_signal"] = (
    (pairs_valid["a"] >= MIN_CASES) & (pairs_valid["PRR"] >= PRR_THRESHOLD) & (pairs_valid["chi_square"] >= CHI2_THRESHOLD)
)
n_signals = pairs_valid["is_signal"].sum()
print(f"\nPairs meeting criteria (PRR>={PRR_THRESHOLD}, chi2>={CHI2_THRESHOLD}, cases>={MIN_CASES}): "
      f"{n_signals:,} of {len(pairs_valid):,}")

q7 = pairs_valid[pairs_valid["is_signal"]].sort_values("PRR", ascending=False).head(10)[
    ["drug_label", "pt_clean", "a", "PRR", "ROR", "chi_square"]
].rename(columns={"a": "co_occurrence_count", "pt_clean": "reaction"}).reset_index(drop=True)
q7[["PRR", "ROR", "chi_square"]] = q7[["PRR", "ROR", "chi_square"]].round(2)

print("\nQ7 (min. 20 cases):")
print(q7.to_string(index=False))
question_results["q7_prr_ror_signals"] = q7


Pairs meeting criteria (PRR>=2, chi2>=4, cases>=20): 3,629 of 1,117,569

Q7 (min. 20 cases):
                            drug_label                             reaction  co_occurrence_count       PRR       ROR  chi_square
         NADOFARAGENE FIRADENOVEC-VNCG          INSTILLATION SITE DISCHARGE                   81 443729.85 612471.80       52.16
               LACTOBACILLUS RHAMNOSUS                  AMYLOID ARTHROPATHY                   27 271827.39 836389.90     1045.61
                              NABILONE       INSPIRATORY CAPACITY DECREASED                   61 125488.76 419902.36       88.86
                       AGALSIDASE BETA   GLOBOTRIAOSYLSPHINGOSINE INCREASED                   65 120287.28 129999.84       41.81
                         OMAVELOXOLONE                  FRIEDREICH'S ATAXIA                   39  79991.43  84173.24       71.35
                               MENTHOL OCCUPATIONAL EXPOSURE TO TOXIC AGENT                  183  78260.62 103386.07       17.03
   

In [ ]:
cache_path = os.path.join(CACHE_DIR, "question_results.pkl")
with open(cache_path, "wb") as f:
    pickle.dump(question_results, f)
print(f"\nCached updated question_results -> {cache_path}")

EXPORT_DIR = os.path.join(CACHE_DIR, "mysql_export")
os.makedirs(EXPORT_DIR, exist_ok=True)
csv_path = os.path.join(EXPORT_DIR, "q7_prr_ror_signals.csv")
q7.to_csv(csv_path, index=False)
print(f"Exported corrected Q7 -> {csv_path}")


Cached updated question_results -> /content/drive/MyDrive/FAERS 2025/_cache/question_results.pkl
Exported corrected Q7 -> /content/drive/MyDrive/FAERS 2025/_cache/mysql_export/q7_prr_ror_signals.csv
